In [2]:
# -*- coding: utf-8 -*-
# =====================================================================================
# 논문 1 / S0. 사전등록·동결 (단일 실행 파일, v4.1)
# v4 -> v4.1 변경은 '★[v4.1-n]' 표시 (v3->v4 표시 ★[변경n]은 유지)
#   ★[v4.1-1] 실행 단계를 STAGE 하나로 통일: "diag"(진단만) -> "draft"(PREREG·게이트 점검) -> "freeze"(동결). sys.exit 제거(노트북 안전)
#   ★[v4.1-2] 창간 전 논문 검토 파일 REVIEW_<저널>_prefounding.csv 자동 생성(decision 열을 EXCLUDE/KEEP으로 채움, 덮어쓰지 않음)
#   ★[v4.1-3] 제외 pid 처리: EXCLUDE 판정 논문은 사건·노출 계산용 코퍼스 F1X에서 제외(F 원본 해시는 그대로, 파생 단계에서 적용)
#   ★[v4.1-4] DOI 접두 분포 점검 DOI_PREFIX_REPORT.csv (저널별 희소 접두 = 오귀속 후보)
#   ★[v4.1-5] fixed_venues 14개 확정 + 규칙 산출 집합과 일치 검사 + 매년 존재 검사
#   ★[v4.1-6] 패치 A: exposure_test 사양을 PREREG에 구체화(primary_fallback)
#   ★[v4.1-7] 패치 B: 주 시나리오가 NONE이어도 S-B(high) 사건에 source_url·pending 게이트 적용
#   ★[v4.1-8] risk_name.variant_names는 'TODO_'로 시작하면 미결로 취급(실제 이름 7개로 교체해야 PREREG.json 저장)
#   ★[v4.1-9] events_override_TEMPLATE.csv 자동 생성(pending 저널), 검토 미결정 행 게이트, 제외 pid·검토 파일 해시 동결
# 사용 순서
#   1) STAGE="diag"  : [0] 진단 + 검토 파일·DOI 접두 보고서·override 템플릿 생성 (아무것도 동결하지 않음)
#   2) 사람이 할 일  : REVIEW_*.csv의 decision 채움, events_catalog.csv에 source_url/retrieved_date 채움(TEMPLATE 참고),
#                      events_override.csv 작성(TEMPLATE 참고), variant_names 7개 입력
#   3) STAGE="draft" + FORCE_PREREG=True(v3->v4 최초 1회): 게이트 PASS/FAIL 확인. FAIL이 없을 때까지 반복(FORCE_PREREG는 PREREG.json이 저장된 뒤 False로)
#   4) STAGE="freeze": 동결. S5 실행 전에 끝낼 것. 동결 직후 git commit + tag
# 주의
#   - 데이터가 없는 환경에서 합성 자료로 [0] 흐름(검토 파일 생성, 제외 반영, 게이트)만 시험했습니다. 실제 F1~F7 전체 실행은 못 했습니다.
#   - 임계값(2%, 5%, 10%, 2000년, 우연 적중률 0.50 등)은 탐색 결과를 본 뒤 정한 값입니다. PREREG.md에 그 사실을 적으세요.
# =====================================================================================
import json, sys, subprocess, hashlib, platform, warnings, traceback
from datetime import datetime
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

# ---------------- 설정 ----------------
CANDS = [Path("../analysis_data/final"), Path("analysis_data/final"), Path("./final"),
         Path.home() / "Research/Minnesota_Transport/analysis_data/final"]
DATA = next((p for p in CANDS if (p / "F1_papers.csv").exists()), None)
assert DATA is not None, f"F1_papers.csv를 찾지 못했습니다. 현재 위치 {Path.cwd()} | 후보 {[str(p) for p in CANDS]}"
ROOT = DATA.resolve().parent.parent
OUT = ROOT / "paper1" / "00_S0"; OUT.mkdir(parents=True, exist_ok=True)
OA_CACHE = DATA.resolve().parent / "openalex" / "works_by_doi.jsonl"
STAGE = "diag"                 # ★[v4.1-1] "diag" | "draft" | "freeze"
FORCE_PREREG = False           # v3->v4 PREREG.json 덮어쓰기 허용(CHANGELOG에 자동 기록). 저장 성공 후 False로 되돌릴 것
FORCE_EVENTS = False           # 이미 동결된 EVENTS_FROZEN.csv 덮어쓰기 허용(사유를 CHANGELOG에 직접 기록)
assert STAGE in ("diag", "draft", "freeze")

FOUND = ["founding", "founding_oa"]
EVENT_GROUPS = {"founding": "A", "founding_oa": "A", "split": "B", "rename": "B", "rename_publisher": "B",
                "publisher_change": "C", "apc_start": "C", "first_issue_oa": "C", "first_jif": "D", "indexing": "D"}
SCENARIOS = {"S-A 창간": "A", "S-B 창간+구조변경": "AB", "S-C 창간+구조+출판환경": "ABC",
             "S-D 평판(첫 JIF·색인)": "D", "S-F 합집합": "ABCDX"}      # S-E(비중 돌파)는 build_events 결과로 따로 계산
SC_KEY = {k.split()[0]: v for k, v in SCENARIOS.items()}

PREREG = dict(
    version="v4", seed=2026, bootstrap_B=2000, fdr_q=0.05,
    weights=["equal", "inv_n_authors", "first_last", "full_count"],
    primary_weight="equal",                                           # 제안값(현 F2 방식). 다르면 수정
    decomposition=dict(primary="griliches_regev_midpoint", sensitivity=["fhk", "price"]),
    k_shrink=5,
    K_sensitivity=dict(K=[75, 150, 300], seeds=5),
    main_window=[2000, 2025], extended_window=[1995, 1999],
    country_cov_min=0.80, country_acc=0.95, wilson_low=0.90, lens_n=500, manual_n=200,
    regime=dict(min_len=3, penalty="BIC", k_max=6, perm_B=1000, trend_sim_B=1000, topic_split_B=200),
    event=dict(
        E1_founding="출판사·서지 기록의 창간(또는 현 제호 개명) 연도: 카탈로그 founding/founding_oa 최소 연도",
        E1_corpus_entry="F1X(제외 pid 반영)에서 저널의 첫 논문 연도",
        entry_event_def="entry_year_ruling(override)이 있으면 그 값, 없으면 E1_corpus_entry. 두 E1이 다르고 판정이 없으면 pending",
        exclude_rule="REVIEW_*_prefounding.csv의 decision==EXCLUDE 및 exclude_pids.csv. 사건·노출 계산과 이후 모든 파생 분석에서 제외(원본 F 파일은 불변)",   # ★[v4.1-3]
        share_thresholds=[0.02, 0.05, 0.10],
        large_event_threshold=0.05, large_event_consecutive_years=2,
        event_min_year=2000, tol_years=1, circ_shift_B=2000,
        cp_range=[2003, 2023],
        fixed_venues=['AAP', 'CACIE', 'JTG', 'T-ITS', 'TR-A', 'TR-B', 'TR-C', 'TR-D', 'TR-E', 'TR-F', 'TRR', 'TransSci', 'TranspPol', 'TranspRev'],   # ★[v4.1-5]
        fixed_venues_rule="F1X 기준 2000~2025 매년 논문이 1편 이상 있는 저널(손으로 추가·삭제 금지)",
        exposure_def="고정 저널 밖 저널들의 연도별 코퍼스 비중 합",
        size_def="첫 임계 돌파 이후 3년 평균 비중 - 직전 연도 비중",
        catalog_file="events_catalog.csv", event_groups=EVENT_GROUPS, scenarios=SCENARIOS,
        catalog_rule="저널 연혁 검색으로 만든 사건 목록(결과를 본 뒤 작성). 게이트 대상 사건은 confidence=high AND source_url 존재, low는 전면 제외",
        primary_candidates=["S-A", "S-B", "S-C"], primary_conf="high",
        sensitivity_scenarios=["S-D", "S-E", "S-F"],
        primary_rule=dict(max_chance_hit=0.50, min_unique_years=6, tol_order=[1, 0],
                          pick="조건을 만족하는 후보 중 고유 사건 연도 수 최대, 동률이면 더 작은 시나리오"),
        primary_fallback="continuous_exposure",
        exposure_test=dict(                                           # ★[v4.1-6] 패치 A (진단에서 NONE을 본 뒤 구체화: 사후 명시)
            series="EXPOSURE_nonfixed.csv: 연도별 비고정 저널 비중 합",
            outcome="d_all(t) - d_fixed(t): 전체 표본과 고정 저널 표본의 주제 분포(topic150 비중) 연간 JS 거리 차 = 구성 유발 변화",
            stat="Spearman(Δexposure(t), outcome(t)), t=2001~2025",
            null="exposure 시계열 순환이동 2000회(자기상관 보존)",
            sensitivity="사건 시나리오 S-A/B/C/E 대조(tol 0과 ±1), 사건 크기 가중"),
        gate_scenario_when_none="S-B",                                # ★[v4.1-7] 패치 B: NONE이어도 이 시나리오(high)의 사건에 게이트 적용
        require_source_url=True),
    grade_A_min_share=0.50, quadrant_stability_min=0.70, multiverse_flip_max=0.25,
    pool=dict(min_active_years=10, min_topic_papers=3, min_regimes=2, strata_min=25),
    risk_name=dict(per_year=12, n_variants=7,
                   variant_names=[f"TODO_V{i}" for i in range(1, 8)]),   # ★[v4.1-8] S2.2의 위험군 변형 7종 실제 이름으로 교체
    periods={"P95_99": [1995, 1999], "P00_04": [2000, 2004], "P05_09": [2005, 2009], "P10_14": [2010, 2014],
             "P12_14": [2012, 2014], "P15_19": [2015, 2019], "P20_24": [2020, 2024], "P22_24": [2022, 2024]},
    n_topics=150,
)
RNG = np.random.default_rng(PREREG["seed"])
hdr = lambda t: print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)
CHECKS = []; STATE = {}
def check(name, cond, detail="", skip=False):
    CHECKS.append(dict(검사=name, 상태="SKIP" if skip else ("PASS" if cond else "FAIL"), 관측=str(detail)))
    return bool(cond) and not skip
def safe(name, fn):
    try: fn()
    except Exception: print(f"✗ [{name}] 실패"); traceback.print_exc(); check(name + " 실행", False, "예외 발생")
sha256 = lambda b: hashlib.sha256(b).hexdigest()
now = lambda: datetime.now().astimezone().isoformat(timespec="seconds")
def changelog(line):
    cl = OUT / "CHANGELOG.md"
    if not cl.exists(): cl.write_text("# CHANGELOG (날짜 | 변경 | 사유 | 영향 받는 결과)\n\n", encoding="utf-8")
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")

# ---------------- 데이터 읽기 ----------------
NAMES = ["F1_papers.csv", "F2_author_paper.csv", "F2_risk_names.csv", "F3_author_period_venue_topic.csv",
         "F4_authors.csv", "F5_continuing_authors.csv", "F6_topics.csv", "F7_oa_author_ids.csv", "MANIFEST.csv"]
RAW = {f: (DATA / f).read_bytes() for f in NAMES}
F1, F2, RISK, F3, F4, F5, F6, F7 = (pd.read_csv(DATA / f, low_memory=False) for f in NAMES[:-1])
MAN = pd.read_csv(DATA / "MANIFEST.csv", dtype=str)
MAN["rows"] = MAN.rows.astype(int); MAN["cols"] = MAN.cols.astype(int)
K = PREREG["n_topics"]
F1X = F1            # ★[v4.1-3] 사건·노출 계산용 코퍼스(제외 pid 반영 후 refresh_exclusion에서 갱신)
if "title" not in F1.columns: F1["title"] = ""
print(f"데이터 폴더: {DATA.resolve()}\n결과 폴더  : {OUT}\n실행 단계  : STAGE={STAGE}")

# ============================================================
# 우연 적중률 함수
# ============================================================
def chance_hit(years, lo, hi, tol):
    """변화점이 [lo,hi]에서 균등하게 하나 떨어질 때, 사건 ±tol 창에 우연히 들어갈 확률"""
    ev = np.unique(np.asarray(years, dtype=int))
    if len(ev) == 0: return np.nan
    ys = np.arange(lo, hi + 1)
    return float(np.mean([(np.abs(y - ev) <= tol).any() for y in ys]))

def expected_event_hit(years, lo, hi, tol, k):
    """변화점 k개가 균등 무작위일 때, 변화점이 닿을 수 있는 사건 중 ±tol 안에 변화점이 따르는 사건의 기대 비율(방향 ii)"""
    ev = np.unique(np.asarray(years, dtype=int)); ys = np.arange(lo, hi + 1); n = len(ys)
    p = [1 - (1 - np.sum(np.abs(ys - e) <= tol) / n) ** k for e in ev if np.sum(np.abs(ys - e) <= tol) > 0]
    return float(np.mean(p)) if p else np.nan

# ============================================================
# 카탈로그·판정·검토·제외·저널 표
# ============================================================
def load_catalog():
    p = OUT / PREREG["event"]["catalog_file"]
    if not p.exists(): print(f"{p.name} 없음"); return None
    C = pd.read_csv(p)
    miss = [c for c in ("source_url", "retrieved_date") if c not in C.columns]
    for c in miss: C[c] = ""
    C["source_url"] = C.source_url.fillna("").astype(str); C["retrieved_date"] = C.retrieved_date.fillna("").astype(str)
    if miss:
        C.to_csv(OUT / "events_catalog_TEMPLATE.csv", index=False, encoding="utf-8-sig")
        print(f"※ 카탈로그에 {miss} 열이 없습니다. events_catalog_TEMPLATE.csv에 열을 추가해 두었으니 채운 뒤 events_catalog.csv로 교체하세요.")
    C["group"] = C.event_type.map(EVENT_GROUPS).fillna("X"); return C

def load_override():
    p = OUT / "events_override.csv"; cols = ["venue_short", "entry_year_ruling", "source_url", "note"]
    if not p.exists(): return pd.DataFrame(columns=cols)
    O = pd.read_csv(p, encoding="utf-8-sig")
    if "E1_confirmed" in O.columns and "entry_year_ruling" not in O.columns: O = O.rename(columns={"E1_confirmed": "entry_year_ruling"})
    for c in cols:
        if c not in O.columns: O[c] = np.nan
    return O[cols]

def doi_prefix(df): return df.doi_norm.astype(str).str.extract(r"^(10\.\d+)")[0]

def doi_prefix_report():                                                  # ★[v4.1-4]
    d = F1.assign(pref=doi_prefix(F1))
    g = d.groupby(["venue_short", "pref"]).size().rename("n").reset_index()
    g["share"] = g.n / g.groupby("venue_short").n.transform("sum")
    yrs = d.groupby(["venue_short", "pref"]).year.agg(["min", "max"]).reset_index().rename(columns={"min": "year_min", "max": "year_max"})
    g = g.merge(yrs, on=["venue_short", "pref"])
    g = g.merge(d.groupby("venue_short").pref.apply(lambda s: s.isna().mean()).rename("doi_missing_share").reset_index(), on="venue_short")
    g.sort_values(["venue_short", "n"], ascending=[True, False]).to_csv(OUT / "DOI_PREFIX_REPORT.csv", index=False, encoding="utf-8-sig")
    rare = g[g.share < 0.05].sort_values("share")
    print("\n(e) DOI 접두 희소 조합(저널 내 점유 5% 미만). 복수 출판사 이력 저널은 정당한 경우도 많으니 후보로만 보세요(전체: DOI_PREFIX_REPORT.csv)")
    print(rare.head(25).to_string(index=False, float_format=lambda x: f"{x:.3f}") if len(rare) else "    없음")

def make_reviews(C):                                                      # ★[v4.1-2]
    """코퍼스 첫 해가 창간보다 앞선 저널의 창간 전 논문 검토 파일을 만든다(이미 있으면 덮어쓰지 않음)"""
    if C is None: return []
    fnd = C[C.event_type.isin(FOUND)].groupby("venue_short").event_year.min().to_dict()
    first = F1.groupby("venue_short").year.min().to_dict()
    d = F1.assign(pref=doi_prefix(F1))
    modal = d.groupby("venue_short").pref.agg(lambda s: s.value_counts().index[0] if s.notna().any() else "").to_dict()
    made = []
    for v, fy in fnd.items():
        if v in first and first[v] < fy:
            p = OUT / f"REVIEW_{v}_prefounding.csv".replace(" ", "_")
            if p.exists(): continue
            pre = d[(d.venue_short == v) & (d.year < fy)][["pid", "venue_short", "year", "title", "doi_norm", "pref"]].rename(columns={"pref": "doi_prefix"})
            pre["venue_modal_prefix"] = modal.get(v, ""); pre["decision"] = ""; pre["reason"] = ""
            pre.sort_values(["year", "pid"]).to_csv(p, index=False, encoding="utf-8-sig"); made.append(p.name)
    return made

def load_reviews():
    fr = []
    for p in sorted(OUT.glob("REVIEW_*_prefounding.csv")):
        R = pd.read_csv(p, dtype=str, encoding="utf-8-sig").fillna(""); R["file"] = p.name; fr.append(R)
    if not fr: return pd.DataFrame(columns=["pid", "venue_short", "year", "title", "decision", "reason", "file"])
    R = pd.concat(fr, ignore_index=True); R["decision"] = R.decision.str.strip().str.upper(); return R

def refresh_exclusion():                                                  # ★[v4.1-3]
    global F1X
    R = load_reviews(); ex = set(R[R.decision == "EXCLUDE"].pid)
    p = OUT / "exclude_pids.csv"
    if p.exists(): ex |= set(pd.read_csv(p, dtype=str, encoding="utf-8-sig").pid)
    STATE["excluded"] = ex; STATE["reviews"] = R
    F1X = F1[~F1.pid.astype(str).isin(ex)].copy()
    return ex

def venue_table(C, O):
    V = F1X.groupby("venue_short").year.min().rename("E1_corpus_entry").reset_index()
    fnd = {} if C is None else C[C.event_type.isin(FOUND)].groupby("venue_short").event_year.min().to_dict()
    V["E1_founding"] = V.venue_short.map(fnd)
    rul = O.dropna(subset=["entry_year_ruling"]).set_index("venue_short").entry_year_ruling.astype(int).to_dict()
    V["entry_ruling"] = V.venue_short.map(rul)
    V["conflict"] = V.E1_founding.notna() & (V.E1_founding != V.E1_corpus_entry)
    V["E1_effective_entry"] = V.entry_ruling.fillna(V.E1_corpus_entry).astype(int)
    V["pending"] = V.conflict & V.entry_ruling.isna()
    return V

def first_cross(s, t, consec):
    run = (s >= t).astype(int).rolling(consec).sum().shift(-(consec - 1))
    hit = run[run >= consec]
    return int(hit.index.min()) if len(hit) else np.nan

def build_events(V):
    ev = PREREG["event"]; ths = ev["share_thresholds"]; lt = ev["large_event_threshold"]; cons = ev["large_event_consecutive_years"]
    assert lt in ths, "large_event_threshold는 share_thresholds 안에 있어야 합니다"
    yr = F1X.groupby(["venue_short", "year"]).size().unstack(fill_value=0)
    yr = yr.reindex(columns=range(int(yr.columns.min()), int(yr.columns.max()) + 1), fill_value=0)
    sh = yr.div(yr.sum(axis=0), axis=1); colname = lambda t: f"E_share{int(round(t * 100))}"; lc = colname(lt)
    rows = []
    for v in yr.index:
        y = yr.loc[v]; s = sh.loc[v]
        r = dict(venue_short=v, n_papers=int(y.sum()), max_share=float(s.max()))
        for t in ths: r[colname(t)] = first_cross(s, t, cons)
        e0 = r[colname(ths[0])]
        r["event_size"] = float(s.loc[int(e0):int(e0) + 2].mean() - s.get(int(e0) - 1, 0.0)) if pd.notna(e0) else np.nan
        e = r[lc]; r["n_papers_at_large"] = int(y.loc[int(e)]) if pd.notna(e) else np.nan
        rows.append(r)
    E = pd.DataFrame(rows).merge(V, on="venue_short", how="left").sort_values("E1_corpus_entry")
    E["is_entry_event"] = E.E1_effective_entry >= ev["event_min_year"]
    E["is_event_eligible"] = E[[colname(t) for t in ths]].ge(ev["event_min_year"]).any(axis=1)
    E["is_large_event"] = E[lc].ge(ev["event_min_year"])
    full = yr.loc[:, 2000:2025].gt(0).all(axis=1); STATE.update(large_col=lc, yr=yr, full_set=set(full[full].index))
    fx = ev["fixed_venues"]
    if fx:
        bad = sorted(set(fx) - set(E.venue_short)); check("fixed_venues가 모두 코퍼스 저널", not bad, bad or "일치")
        gap = {v: [int(c) for c in range(2000, 2026) if yr.loc[v, c] == 0] for v in fx if v in yr.index and (yr.loc[v, 2000:2025] == 0).any()}
        check("fixed_venues가 2000~2025 매년 논문 존재(F1X 기준)", not gap, gap or "모두 존재")   # ★[v4.1-5]
        diff = sorted(set(fx) ^ STATE["full_set"]); check("fixed_venues = 규칙 산출 집합", not diff, diff or "일치")
        expo = sh.loc[~sh.index.isin(fx)].sum(axis=0).rename("nonfixed_share").reset_index()
        expo.to_csv(OUT / "EXPOSURE_nonfixed.csv", index=False, encoding="utf-8-sig"); print(f"연속 노출 시계열 저장 ({len(expo)}개 연도)")
    return E

def scenario_events(C, V):
    ev = PREREG["event"]; P = C.copy()
    eff = V.set_index("venue_short").entry_ruling.dropna().astype(int).to_dict()
    m = P.event_type.isin(FOUND) & P.venue_short.isin(eff)
    P.loc[m, "event_year"] = P.loc[m, "venue_short"].map(eff)
    P["event_year"] = P.event_year.astype(int)
    return P[P.event_year.between(ev["event_min_year"], 2025) & (P.confidence != "low") & P.venue_short.isin(V.venue_short)]

def chance_table(POST, E):
    ev = PREREG["event"]; lo, hi = ev["cp_range"]; rows = []
    def mk(key, desc, conf, yrs):
        r = dict(scenario=key, 설명=desc, conf=conf, n_events=len(yrs), n_years=len(set(yrs)),
                 chance_tol0=chance_hit(yrs, lo, hi, 0), chance_tol1=chance_hit(yrs, lo, hi, 1))
        for k in (2, 3, 4, 5): r[f"exp_ii_k{k}"] = expected_event_hit(yrs, lo, hi, 1, k)
        return r
    for conf in ("high", "high+medium"):
        D = POST[POST.confidence == "high"] if conf == "high" else POST
        for name, g in SCENARIOS.items():
            yrs = D[D.group.isin(list(g))].event_year.astype(int).tolist(); rows.append(mk(name.split()[0], name, conf, yrs))
    big = E[E.is_large_event][STATE["large_col"]].astype(int).tolist()
    rows.append(mk("S-E", f"비중 돌파({int(round(ev['large_event_threshold'] * 100))}%, 연속 {ev['large_event_consecutive_years']}년)", "n/a", big))
    return pd.DataFrame(rows)

def select_primary(T):
    ev = PREREG["event"]; r = ev["primary_rule"]
    for tol in r["tol_order"]:
        d = T[(T.conf == ev["primary_conf"]) & T.scenario.isin(ev["primary_candidates"])
              & (T[f"chance_tol{tol}"] <= r["max_chance_hit"]) & (T.n_years >= r["min_unique_years"])]
        if len(d):
            d = d.sort_values(["n_years", "scenario"], ascending=[False, True]).iloc[0]
            return dict(primary=d.scenario, tol=int(tol), n_years=int(d.n_years), chance=float(d[f"chance_tol{tol}"]))
    return dict(primary="NONE", tol=None, fallback=ev["primary_fallback"])

def open_items():
    ev = PREREG["event"]; o = []
    if not ev["fixed_venues"]: o.append("event.fixed_venues")
    vn = PREREG["risk_name"]["variant_names"]
    if not vn or any(str(x).startswith("TODO_") for x in vn) or len(vn) != PREREG["risk_name"]["n_variants"]: o.append("risk_name.variant_names")
    return o

# ============================================================
# [0] 진단: 검토 파일, E1 충돌, 고정 저널, 우연 적중률, 주 시나리오
# ============================================================
def sec0():
    hdr("[0] 진단: 오귀속 검토 -> E1 충돌 -> 시나리오별 우연 적중률 -> 주 시나리오 결정")
    ev = PREREG["event"]; lo, hi = ev["cp_range"]
    C = load_catalog(); O = load_override()
    made = make_reviews(C)
    if made: print(f"※ 검토 파일 생성: {made}  -> decision 열에 EXCLUDE(제외) 또는 KEEP(정당)을 채운 뒤 다시 실행하세요.")
    ex = refresh_exclusion(); R = STATE["reviews"]
    print(f"제외 pid {len(ex)}편 반영(F1 {len(F1):,} -> F1X {len(F1X):,})")
    V = venue_table(C, O); E = build_events(V); STATE.update(C=C, O=O, V=V, E=E)
    if C is None: print("카탈로그가 없어 [0]을 중단합니다."); return
    # (a) 충돌 점검
    print("\n(a) 창간 연도 vs 코퍼스 첫 논문 연도(제외 반영 후) 불일치 저널")
    if not V.conflict.any(): print("    없음")
    for _, r in V[V.conflict].iterrows():
        v = r.venue_short; early = r.E1_corpus_entry < r.E1_founding
        print(f" - {v}: 창간 {int(r.E1_founding)} / 코퍼스 {int(r.E1_corpus_entry)} / 판정 {int(r.entry_ruling) if pd.notna(r.entry_ruling) else '없음(pending)'}"
              f" -> {'코퍼스가 창간보다 앞섬(검토 파일 확인)' if early else '코퍼스가 창간보다 늦음(범위 편입 시점)'}")
    if len(R):
        s = R.groupby("venue_short").decision.value_counts().unstack(fill_value=0)
        print("\n    검토 파일 판정 현황(저널별):\n" + s.to_string())
    pend = V[V.pending]
    if len(pend):
        T0 = pend[["venue_short"]].copy(); T0["entry_year_ruling"] = ""; T0["source_url"] = ""
        T0["note"] = [f"코퍼스 첫 해 {int(a)}, 창간 {int(b)}" for a, b in zip(pend.E1_corpus_entry, pend.E1_founding)]
        T0.to_csv(OUT / "events_override_TEMPLATE.csv", index=False, encoding="utf-8-sig")
        print(f"    pending {pend.venue_short.tolist()} -> events_override_TEMPLATE.csv 저장(entry_year_ruling=코퍼스 진입 사건으로 쓸 연도, source_url 필수)")
    # (b) 고정 저널
    print(f"\n(b) 2000~2025 매년 논문이 있는 저널(F1X 기준) {len(STATE['full_set'])}개: {sorted(STATE['full_set'])}")
    print(f"    PREREG.fixed_venues {len(ev['fixed_venues'])}개와 {'일치' if set(ev['fixed_venues']) == STATE['full_set'] else '불일치: ' + str(sorted(set(ev['fixed_venues']) ^ STATE['full_set']))}")
    # (c) 우연 적중률
    POST = scenario_events(C, V); T = chance_table(POST, E); PRIM = select_primary(T); STATE.update(POST=POST, T=T, PRIM=PRIM)
    print(f"\n(c) 변화점 후보 연도 {lo}~{hi} ({hi - lo + 1}개). chance_tolX = 변화점 1개가 우연히 사건 ±X년 창에 들어갈 확률")
    print("    exp_ii_kN = 변화점 N개가 무작위일 때 변화점이 닿을 수 있는 사건 중 ±1년 내 변화점이 따르는 사건의 기대 비율")
    print(T.drop(columns="설명").to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    r = ev["primary_rule"]
    print(f"\n(d) 주 시나리오 결정 규칙: 후보 {ev['primary_candidates']} / conf={ev['primary_conf']} / tol 순서 {r['tol_order']} / 우연 적중률 ≤ {r['max_chance_hit']} / 고유 연도 ≥ {r['min_unique_years']}")
    print("    결과:", PRIM)
    if PRIM["primary"] == "NONE":
        print("    -> 주 분석은 연속 노출 시계열(PREREG.event.exposure_test), 사건 시나리오는 민감도. 게이트는 S-B(high) 사건에 적용.")
    else: print(f"    -> 주 시나리오 {PRIM['primary']}, 허용오차 ±{PRIM['tol']}년.")
    if (T[T.scenario.isin(["S-E", "S-F"])].chance_tol1 > 0.8).any(): print("    ⚠ S-E 또는 S-F의 우연 적중률이 0.8을 넘으면 그 시나리오의 일치 검정은 사실상 정보가 없습니다.")
    T.to_csv(OUT / "SCENARIO_CHANCE.csv", index=False, encoding="utf-8-sig")
    (OUT / "PRIMARY_SCENARIO_DRAFT.json").write_text(json.dumps(PRIM, ensure_ascii=False, indent=2), encoding="utf-8")
    doi_prefix_report()
    print("\n저장: SCENARIO_CHANCE.csv, PRIMARY_SCENARIO_DRAFT.json, DOI_PREFIX_REPORT.csv")
safe("sec0", sec0)

# ============================================================
# [1] PREREG 저장
# ============================================================
def sec1():
    hdr("[1] PREREG 저장")
    body = json.dumps(PREREG, ensure_ascii=False, indent=2, sort_keys=True).encode("utf-8")
    (OUT / "PREREG_DRAFT.json").write_bytes(body)
    STATE["prereg_saved"] = False
    oi = open_items()
    if oi:
        print(f"⚠ 미결 항목 {oi}: PREREG.json은 저장하지 않고 PREREG_DRAFT.json만 저장했습니다.")
        check("PREREG 미결 항목 없음", False, oi); return
    p = OUT / "PREREG.json"
    if p.exists() and sha256(p.read_bytes()) != sha256(body):
        if not FORCE_PREREG:
            print("⚠ 기존 PREREG.json과 내용이 다릅니다. v3->v4 최초 변경이면 FORCE_PREREG=True로 한 번 실행하세요.")
            check("PREREG 변경 없음", False, "기존 파일과 불일치"); return
        changelog(f"- {now()} | PREREG {sha256(p.read_bytes())[:12]} -> v4 {sha256(body)[:12]} | "
                  "(1) 큰 사건(5% 연속 2년 돌파) 6개로 ±1년 대조 변별력 부족(우연 적중률 진단 추가), (2) 주 시나리오 규칙 명시, 진단 결과 NONE -> 연속 노출을 주 분석으로 하는 exposure_test를 "
                  "진단 결과를 본 뒤 구체화(사후 명시), 사건 시나리오는 민감도, (3) E1을 창간/코퍼스 진입으로 분리, (4) 게이트 강화(high+source_url, pending 충돌 차단, NONE일 때 S-B에 적용), "
                  "(5) 큰 사건 규칙을 계획서의 '증가폭 상위 50%'에서 '5% 연속 2년 돌파'로 변경(v3부터), (6) fixed_venues 14개(규칙: 매년 존재)·k_shrink·분해식·K 민감도·위험군 변형 확정, "
                  "(7) T-ITS 등 창간 전 논문 검토와 제외 pid 규칙 추가 | 사건 대조(S5.6), 노출 분석, 갱신 분해(S6)")
    p.write_bytes(body); STATE["prereg_saved"] = True
    print(f"저장: {p.name} | sha256 {sha256(body)[:16]}"); check("PREREG 저장", True, sha256(body)[:16])

# ============================================================
# [2] 데이터·환경 동결
# ============================================================
def sec2():
    hdr("[2] F 파일 해시와 환경 동결")
    rows = [dict(file=f, bytes=len(b), rows=len(d), cols=d.shape[1], sha256=sha256(b), md5_10=hashlib.md5(b).hexdigest()[:10])
            for (f, b), d in zip(RAW.items(), [F1, F2, RISK, F3, F4, F5, F6, F7, MAN])]
    T = pd.DataFrame(rows); T.to_csv(OUT / "MANIFEST_FROZEN.csv", index=False); print(T[["file", "rows", "cols", "md5_10"]].to_string(index=False))
    mm = MAN.set_index("file").md5.to_dict(); bom = b"\xef\xbb\xbf"
    bad = [r["file"] for r in rows if r["file"] in mm and r["md5_10"] != mm[r["file"]]
           and hashlib.md5(bom + RAW[r["file"]]).hexdigest()[:10] != mm[r["file"]]]
    check("F 파일이 MANIFEST와 일치(BOM 차이는 허용)", not bad, bad or "모두 일치")
    try: freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=120).stdout
    except Exception as e: freeze = f"# pip freeze 실패: {e!r}\n"
    (OUT / "requirements_frozen.txt").write_text(freeze, encoding="utf-8")
    try: git = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=ROOT, timeout=20).stdout.strip() or None
    except Exception: git = None
    env = dict(time=now(), python=sys.version, platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__, seed=PREREG["seed"], git_commit=git)
    (OUT / "ENV.json").write_text(json.dumps(env, ensure_ascii=False, indent=2), encoding="utf-8"); print("환경:", env)
    if git is None: print("※ git 커밋 해시가 없습니다. 동결 직후 커밋과 태그(또는 OSF 등록)를 남겨 독립 타임스탬프를 확보하세요.")

# ============================================================
# [3] 사건 목록 초안·게이트·동결
# ============================================================
def sec3():
    hdr("[3] 사건 목록 (초안, 게이트, 동결)")
    ev = PREREG["event"]; E, C, V, O, POST, T, PRIM, R = (STATE[k] for k in ("E", "C", "V", "O", "POST", "T", "PRIM", "reviews")); lc = STATE["large_col"]
    E.to_csv(OUT / "EVENTS_DRAFT.csv", index=False, encoding="utf-8-sig")
    print(E[["venue_short", "E1_corpus_entry", "E1_founding", "entry_ruling", "E1_effective_entry", "pending", "E_share2", lc, "n_papers_at_large",
             "is_entry_event", "is_large_event"]].to_string(index=False))
    print(f"\n큰 사건 {int(E.is_large_event.sum())}개 | 그중 진입 사건 {int((E.is_large_event & E.is_entry_event).sum())}개 | pending 충돌 저널 {V[V.pending].venue_short.tolist()}")
    print(T.drop(columns="설명").to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    # 게이트 ★[v4.1-7] 주 시나리오가 NONE이어도 gate_scenario_when_none의 high 사건에 적용
    gsc = PRIM["primary"] if PRIM["primary"] != "NONE" else ev["gate_scenario_when_none"]
    rows = POST[POST.group.isin(list(SC_KEY[gsc])) & (POST.confidence == "high")]
    print(f"\n게이트 대상: {gsc}(high) 사건 {len(rows)}건, 저널 {rows.venue_short.nunique()}개")
    g1 = check("게이트 대상 사건이 모두 high", (rows.confidence == "high").all(), f"{len(rows)}건")
    src_missing = rows[rows.source_url.str.len() == 0]
    g2 = check("게이트 대상 사건의 source_url 존재", (not ev["require_source_url"]) or len(src_missing) == 0,
               f"{len(src_missing)}건 누락: {sorted(set(src_missing.venue_short))}" if len(src_missing) else "모두 존재")
    pend = sorted(set(V[V.pending].venue_short) & (set(rows.venue_short) | set(E[E.is_large_event].venue_short)))
    g3 = check("게이트 대상·큰 사건에 판정 없는 E1 충돌 저널 없음", not pend, pend or "없음")
    over = O.dropna(subset=["entry_year_ruling"]); ov_nosrc = over[over.source_url.fillna("").astype(str).str.len() == 0].venue_short.tolist()
    g4 = check("판정(override)마다 source_url 존재", not ov_nosrc, ov_nosrc or "모두 존재")
    g5 = check("PREREG.json 저장됨(미결 항목 없음)", STATE.get("prereg_saved", False), open_items() or "저장됨")
    und = R[~R.decision.isin(["EXCLUDE", "KEEP"])] if len(R) else R
    g6 = check("검토 파일의 모든 행이 EXCLUDE/KEEP으로 판정됨", len(und) == 0, f"미결정 {len(und)}행: {sorted(set(und.file)) if len(und) else ''}")   # ★[v4.1-9]
    g7 = check("제외 pid가 모두 F1에 존재", STATE["excluded"] <= set(F1.pid.astype(str)), len(STATE["excluded"]))
    ok = all([g1, g2, g3, g4, g5, g6, g7])
    if STAGE != "freeze": print("→ 초안만 저장했습니다. 게이트가 모두 PASS인지 확인한 뒤 STAGE='freeze'로 다시 실행하세요."); return
    fp = OUT / "EVENTS_FROZEN.csv"
    if fp.exists() and not FORCE_EVENTS: print("이미 동결된 EVENTS_FROZEN.csv가 있어 덮어쓰지 않습니다."); check("사건 목록 신규 동결", False, "기존 동결 파일 존재"); return
    if not ok: print("✗ 게이트 실패로 동결하지 않았습니다."); check("사건 목록 동결", False, "게이트 실패"); return
    exc = F1[F1.pid.astype(str).isin(STATE["excluded"])][["pid", "venue_short", "year", "title", "doi_norm"]]
    files = {"EVENTS_FROZEN.csv": E.to_csv(index=False).encode("utf-8-sig"),
             "EVENTS_CATALOG_FROZEN.csv": (OUT / ev["catalog_file"]).read_bytes(),
             "SCENARIO_CHANCE_FROZEN.csv": T.to_csv(index=False).encode("utf-8-sig"),
             "PRIMARY_SCENARIO_FROZEN.json": json.dumps(PRIM, ensure_ascii=False, indent=2).encode("utf-8"),
             "EXCLUDE_PIDS_FROZEN.csv": exc.to_csv(index=False).encode("utf-8-sig")}
    if (OUT / "events_override.csv").exists(): files["EVENTS_OVERRIDE_FROZEN.csv"] = (OUT / "events_override.csv").read_bytes()
    for p in sorted(OUT.glob("REVIEW_*_prefounding.csv")): files[p.name.replace("REVIEW_", "REVIEW_FROZEN_")] = p.read_bytes()
    logs = []
    for name, b in files.items():
        (OUT / name).write_bytes(b); logs.append(dict(file=name, sha256=sha256(b), frozen_at=now(), seed=PREREG["seed"]))
    log = pd.DataFrame(logs); log.to_csv(OUT / "FREEZE_LOG.csv", mode="a", header=not (OUT / "FREEZE_LOG.csv").exists(), index=False)
    print(log[["file", "sha256", "frozen_at"]].to_string(index=False)); check("사건 목록 동결", True, logs[0]["sha256"][:16])
    print("→ 지금 git commit + tag(또는 OSF 등록)로 독립 타임스탬프를 남기세요. 이후 F 파일은 원본 그대로, 분석은 F1X(제외 pid 반영)로 하세요.")

# ============================================================
# [4] 불변식 검증 (원본 F 파일 기준, v3과 동일)
# ============================================================
def sec4():
    hdr("[4] 불변식(원본 F 파일 기준)")
    check("F1 pid 유일", not F1.pid.duplicated().any())
    check("F1 연도 범위 1967~2025, in_sample 전부 True", F1.year.between(1967, 2025).all() and F1.in_sample.astype(bool).all(), (F1.year.min(), F1.year.max()))
    check("F2 키(pid,name_norm) 유일", not F2.duplicated(["pid", "name_norm"]).any())
    pw = F2.groupby("pid").w.sum(); check("F2 논문당 w 합 = 1", np.allclose(pw, 1), f"max|dev|={np.abs(pw - 1).max():.2e}")
    pr = F2.groupby("pid").size(); check("F2 w = 1/(논문의 F2 행 수)", np.allclose(F2.w, 1 / F2.pid.map(pr)))
    check("F2.pid ⊂ F1.pid", F2.pid.isin(F1.pid).all())
    m = F2.merge(F1[["pid", "year", "venue_short", "n_authors", "topic150"]], on="pid", suffixes=("", "_f1"))
    check("F2의 year·venue·n_authors·topic150 = F1", all((m[c] == m[c + "_f1"]).all() for c in ["year", "venue_short", "n_authors", "topic150"]))
    check("위험군 이름이 F2·F3·F4·F5에 없음", not any(d.name_norm.isin(RISK.name_norm).any() for d in (F2, F3, F4, F5)))
    ok = True; det = {}
    for p, (a, b) in PREREG["periods"].items():
        mass = F3.loc[F3.period == p, "w"].sum(); n = F2[F2.year.between(a, b)].pid.nunique(); det[p] = (round(mass, 3), n); ok &= np.isclose(mass, n)
    check("F3 기간 질량 합 = F2 논문 수", ok, det)
    rb = pd.concat([F2[F2.year.between(a, b)].groupby(["name_norm", "venue_short", "topic150"]).w.sum().reset_index().assign(period=p)
                    for p, (a, b) in PREREG["periods"].items()])
    mm = F3.merge(rb, on=["name_norm", "period", "venue_short", "topic150"], how="outer", suffixes=("", "_re"), indicator=True)
    check("F3 = F2 재집계(행·값)", (mm._merge == "both").all() and np.allclose(mm.w, mm.w_re))
    cont = F4[(F4.m_P10_14 > 0) & (F4.m_P20_24 > 0)]
    check("F4 continuing_main이 질량 조건과 일치", ((F4.m_P10_14 > 0) & (F4.m_P20_24 > 0) == F4.continuing_main).all())
    check("F5 저자 = F4 지속 저자", set(F5.name_norm) == set(cont.name_norm))
    check("F1 topic150 ⊂ F6, 주제 수 = 150", F1.topic150.dropna().astype(int).isin(F6.topic150).all() and len(F6) == K)
    sh = lambda a, b: (lambda s: np.bincount(s.topic150, minlength=K) / len(s))(F2[F2.year.between(a, b)].drop_duplicates("pid"))
    check("F6 share_0509·1519·2025 재계산 일치", np.allclose(sh(2005, 2009), F6.share_0509) and np.allclose(sh(2015, 2019), F6.share_1519) and np.allclose(sh(2020, 2025), F6.share_2025))
    check("F6 emerging·declining이 정의(3배·0.5%)로 재현", ((((sh(2015, 2019) + 1e-4) / (sh(2005, 2009) + 1e-4) >= 3) & (sh(2015, 2019) >= .005)) == F6.emerging.values).all())
    check("F7 rank가 논문별 0부터 연속", (F7.groupby("pid")["rank"].max() + 1 == F7.groupby("pid").size()).all())

# ============================================================
# [5] 음성 대조
# ============================================================
def author_concentration(df, min_mass=2.0):
    """가중 질량(w 합) min_mass 이상 저자의 (최대 토픽 질량 합 / 총 질량 합). 논문 편수 기준이 아님"""
    g = df.groupby(["name_norm", "topic150"]).w.sum().reset_index()
    g["tot"] = g.groupby("name_norm").w.transform("sum"); g["mx"] = g.groupby("name_norm").w.transform("max")
    a = g.drop_duplicates("name_norm"); a = a[a.tot >= min_mass]
    return float(a.mx.sum() / a.tot.sum()), len(a)

def sec5(B=20):
    hdr("[5] 음성 대조 (라벨 섞기)")
    d = F2[F2.year.between(2010, 2024)].copy(); real, na = author_concentration(d)
    paper = d.drop_duplicates("pid")[["pid", "venue_short", "year", "topic150"]].reset_index(drop=True)
    grp = paper.groupby(["venue_short", "year"]).indices; sims = []; agg_dev = []
    base_share = np.bincount(paper.topic150, minlength=K) / len(paper)
    for _ in range(B):
        perm = paper.topic150.values.copy()
        for idx in grp.values(): perm[idx] = RNG.permutation(perm[idx])
        dd = d.assign(topic150=d.pid.map(dict(zip(paper.pid, perm)))); sims.append(author_concentration(dd)[0])
        agg_dev.append(np.abs(np.bincount(perm, minlength=K) / len(paper) - base_share).max())
    sims = np.array(sims)
    print(f"저자 주제 집중도(가중 질량 2 이상 저자 {na:,}명): 실제 {real:.4f} | 섞은 후 평균 {sims.mean():.4f} (sd {sims.std():.4f})")
    print(f"집계 주제 분포 최대 편차(섞기 전후): {max(agg_dev):.2e}")
    check("음성 대조: 섞으면 저자 수준 집중이 사라짐(실제 > 섞은 값 최댓값)", real > sims.max(), f"실제 {real:.3f} vs 섞은 {sims.mean():.3f}±{sims.std():.3f}")
    check("음성 대조: 섞어도 집계 분포는 보존", max(agg_dev) < 1e-12, f"{max(agg_dev):.1e}")
    print("※ z값은 의미가 없으니 논문에 쓰지 마세요.")

# ============================================================
# [6] 외부 대조
# ============================================================
def _cites(r):
    c = r.get("cited_by_count")
    if c is not None: return c, "total"
    v = r.get("cby")
    if isinstance(v, (int, float)): return v, "cby_scalar"
    if isinstance(v, dict): return sum(float(x) for x in v.values()), "cby_sum"
    if isinstance(v, list):
        try: return sum(float(x.get("cited_by_count", 0)) for x in v), "cby_sum"
        except Exception: return None, None
    return None, None

def sec6():
    hdr("[6] 외부 대조 (OpenAlex 캐시)")
    name = "외부 대조: 인용 순위상관 ≥ 0.90(임의 기준)"
    if not OA_CACHE.exists(): print(f"캐시 없음: {OA_CACHE}"); check(name, False, "캐시 없음", skip=True); return
    rows = []; kinds = set(); first_keys = None
    with OA_CACHE.open(encoding="utf-8") as f:
        for line in f:
            r = json.loads(line); first_keys = first_keys or list(r.keys())
            c, k = _cites(r)
            if c is None: continue
            kinds.add(k); rows.append((str(r.get("doi_norm") or r.get("doi") or "").lower(), c))
    if not rows: print(f"인용 필드를 해석하지 못했습니다. 캐시 필드: {first_keys}"); check(name, False, "필드 없음", skip=True); return
    O = pd.DataFrame(rows, columns=["doi", "oa_cit"]).drop_duplicates("doi")
    M = F1.assign(doi=F1.doi_norm.str.lower()).dropna(subset=["doi"]).merge(O, on="doi")
    rho_all = M[["cited_by_count", "oa_cit"]].corr(method="spearman").iloc[0, 1]
    M16 = M[M.year >= 2016]; rho16 = M16[["cited_by_count", "oa_cit"]].corr(method="spearman").iloc[0, 1]
    print(f"매칭 {len(M):,}편 | 필드 해석 {kinds} | Spearman 전체 {rho_all:.3f} | 2016년 이후 {rho16:.3f}")
    if "cby_sum" in kinds: print("※ counts_by_year 합산은 제공 연도 범위 밖 인용이 빠져 구 논문에서 과소집계됩니다. 판정은 2016년 이후 부분표본 기준(범위 확인 후 조정).")
    check(name, (rho16 if "cby_sum" in kinds else rho_all) >= 0.90, f"전체 {rho_all:.3f}, 2016+ {rho16:.3f}")

# ============================================================
# 실행 흐름 ★[v4.1-1]
# ============================================================
if STAGE == "diag":
    print("\nSTAGE='diag': [0] 진단만 실행했습니다(PREREG·동결 없음). REVIEW_*.csv 판정, 카탈로그 source_url, override, variant_names를 채운 뒤 STAGE='draft'로 실행하세요.")
else:
    for n, f in (("sec1", sec1), ("sec2", sec2), ("sec3", sec3), ("sec4", sec4), ("sec5", sec5), ("sec6", sec6)): safe(n, f)
    hdr("[7] 검증 종합")
    LOG = pd.DataFrame(CHECKS); LOG.index += 1
    LOG.to_csv(OUT / "S0_checks.csv", encoding="utf-8-sig"); print(LOG[["검사", "상태"]].to_string())
    n_pass, n_fail, n_skip = (int((LOG.상태 == s).sum()) for s in ("PASS", "FAIL", "SKIP"))
    if not (OUT / "CHANGELOG.md").exists(): changelog(f"- {now()} | S0 초기화 | PREREG v4 | 없음")
    print(f"\n검사 {len(LOG)}건: PASS {n_pass}, FAIL {n_fail}, SKIP {n_skip} (SKIP은 '완료'가 아님)")
    if n_skip: print("SKIP 항목:", LOG[LOG.상태 == "SKIP"].검사.tolist())
    if n_fail: print("⚠ FAIL을 해결하기 전에는 S1 이후로 진행하지 마세요(해결·변경은 CHANGELOG에 기록).")
    print(f"산출물: {OUT}")

데이터 폴더: /home/yjlee/Research/Minnesota_Transport/analysis_data/final
결과 폴더  : /home/yjlee/Research/Minnesota_Transport/paper1/00_S0
실행 단계  : STAGE=diag

[0] 진단: 오귀속 검토 -> E1 충돌 -> 시나리오별 우연 적중률 -> 주 시나리오 결정
※ 카탈로그에 ['source_url', 'retrieved_date'] 열이 없습니다. events_catalog_TEMPLATE.csv에 열을 추가해 두었으니 채운 뒤 events_catalog.csv로 교체하세요.
※ 검토 파일 생성: ['REVIEW_AMAR_prefounding.csv', 'REVIEW_T-ITS_prefounding.csv']  -> decision 열에 EXCLUDE(제외) 또는 KEEP(정당)을 채운 뒤 다시 실행하세요.
제외 pid 0편 반영(F1 111,685 -> F1X 111,685)
연속 노출 시계열 저장 (59개 연도)

(a) 창간 연도 vs 코퍼스 첫 논문 연도(제외 반영 후) 불일치 저널
 - AMAR: 창간 2014 / 코퍼스 2013 / 판정 없음(pending) -> 코퍼스가 창간보다 앞섬(검토 파일 확인)
 - RailEng: 창간 1993 / 코퍼스 2020 / 판정 없음(pending) -> 코퍼스가 창간보다 늦음(범위 편입 시점)
 - T-ITS: 창간 2000 / 코퍼스 1997 / 판정 없음(pending) -> 코퍼스가 창간보다 앞섬(검토 파일 확인)
 - TR-E: 창간 1972 / 코퍼스 1997 / 판정 없음(pending) -> 코퍼스가 창간보다 늦음(범위 편입 시점)
 - TRR: 창간 1963 / 코퍼스 1967 / 판정 없음(pending) -> 코퍼스가 창간보다 늦음(범위 편입 시점)

    검토 파일 판정 현황(저널별):
decision      
venue_short   
AMAR         6
T-ITS    

In [3]:
#

In [4]:
import requests, time
# 1) 희소 접두 검토 파일(한 번만 생성, 파일명은 로더 패턴을 따름)
d = F1.assign(doi_prefix=doi_prefix(F1))
d["n_pref"] = d.groupby(["venue_short", "doi_prefix"]).pid.transform("size")
d["share"] = d.n_pref / d.groupby("venue_short").pid.transform("size")
modal = d.groupby("venue_short").doi_prefix.agg(lambda s: s.value_counts().index[0])
rare = d[d.doi_prefix.notna() & (d.share < 0.005) & (d.n_pref <= 10)].copy()
rare["venue_modal_prefix"] = rare.venue_short.map(modal); rare["decision"] = ""; rare["reason"] = ""
p = OUT / "REVIEW_RAREPREFIX_prefounding.csv"
if not p.exists():
    rare[["pid", "venue_short", "year", "title", "doi_norm", "doi_prefix", "venue_modal_prefix", "decision", "reason"]] \
        .sort_values(["venue_short", "year"]).to_csv(p, index=False, encoding="utf-8-sig")

# 2) Crossref 게재지 확인(검토 파일마다 한 번만)
def cr(doi):
    try:
        r = requests.get("https://api.crossref.org/works/" + requests.utils.quote(doi, safe="/"),
                         params={"mailto": "YOUR_EMAIL"}, timeout=30)
        if r.status_code != 200: return dict(cr_status=str(r.status_code))
        m = r.json()["message"]; dp = (m.get("issued", {}).get("date-parts") or [[None]])[0]
        return dict(cr_status="200", cr_container=" | ".join(m.get("container-title", [])), cr_type=m.get("type", ""),
                    cr_year=dp[0] if dp else None, cr_issn=",".join(m.get("ISSN", [])))
    except Exception as e:
        return dict(cr_status=repr(e)[:80])

for f in sorted(OUT.glob("REVIEW_*_prefounding.csv")):
    R = pd.read_csv(f, dtype=str, encoding="utf-8-sig").fillna("")
    if "cr_status" in R.columns: continue
    X = []
    for doi in R.doi_norm:
        X.append(cr(doi) if doi else dict(cr_status="no_doi")); time.sleep(0.2)
    pd.concat([R, pd.DataFrame(X).fillna("")], axis=1).to_csv(f, index=False, encoding="utf-8-sig")
    print(f.name, len(R), "행에 Crossref 정보 추가")

REVIEW_AMAR_prefounding.csv 6 행에 Crossref 정보 추가
REVIEW_RAREPREFIX_prefounding.csv 23 행에 Crossref 정보 추가
REVIEW_T-ITS_prefounding.csv 9 행에 Crossref 정보 추가


In [5]:
#

In [6]:
import re
norm = lambda s: re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()
RV = {f: pd.read_csv(f, dtype=str, encoding="utf-8-sig").fillna("") for f in sorted(OUT.glob("REVIEW_*_prefounding.csv"))}
for f, R in RV.items(): print(f.name, "Crossref 상태:", R.cr_status.value_counts().to_dict())

# 저널별 기준 게재지: 해당 저널의 최빈 접두 논문 8편을 Crossref로 조회해 최빈 container-title 사용
pref = doi_prefix(F1); ref = {}
for v in sorted(set().union(*[set(R.venue_short) for R in RV.values()])):
    pool = F1[(F1.venue_short == v) & (F1.year >= 2005) & F1.doi_norm.notna()]
    pool = pool[pref.loc[pool.index] == pref[F1.venue_short == v].value_counts().index[0]]
    cs = []
    for doi in pool.sample(min(8, len(pool)), random_state=2026).doi_norm:
        cs.append(norm(cr(doi).get("cr_container", ""))); time.sleep(0.2)
    cs = [c for c in cs if c]; ref[v] = pd.Series(cs).value_counts().index[0] if cs else ""
print("기준 게재지:", ref)

# suggest 열은 참고용입니다. decision은 직접 채우세요.
for f, R in RV.items():
    R["ref_container"] = R.venue_short.map(ref)
    match = np.array([a != "" and norm(a) == b and b != "" for a, b in zip(R.cr_container, R.ref_container)])
    ok = (R.cr_status == "200").values
    if "RAREPREFIX" in f.name: sug = np.where(~ok, "REVIEW", np.where(match, "KEEP?", "EXCLUDE?"))
    else: sug = np.where(~ok, "REVIEW", np.where(match, "CHECK(창간 전인데 게재지 일치)", "EXCLUDE?"))
    R["suggest"] = sug; R.to_csv(f, index=False, encoding="utf-8-sig")
    print("\n" + f.name)
    print(R[["venue_short", "year", "cr_year", "title", "cr_container", "cr_type", "suggest"]]
          .assign(title=R.title.str[:50], cr_container=R.cr_container.str[:45]).to_string(index=False))

REVIEW_AMAR_prefounding.csv Crossref 상태: {'200': 6}
REVIEW_RAREPREFIX_prefounding.csv Crossref 상태: {'404': 22, '200': 1}
REVIEW_T-ITS_prefounding.csv Crossref 상태: {'no_doi': 8, '200': 1}
기준 게재지: {'AAP': 'accident analysis amp prevention', 'AMAR': 'analytic methods in accident research', 'CACIE': 'computer aided civil and infrastructure engineering', 'ETRR': 'european transport research review', 'JPT': 'journal of public transportation', 'JTTE': 'journal of traffic and transportation engineering english edition', 'T-ITS': 'ieee transactions on intelligent transportation systems', 'TRIP': 'transportation research interdisciplinary perspectives', 'TRR': 'transportation research record journal of the transportation research board', 'TransSci': 'transportation science'}

REVIEW_AMAR_prefounding.csv
venue_short year cr_year                                              title                          cr_container         cr_type              suggest
       AMAR 2013    2014 Analytic methods in

In [7]:
#

In [ ]:
# ## 지금까지 한 일
# **목표:** 논문 1(교통 지식 생산의 체제 전환)의 S0 사전등록·동결 단계. 사건 목록과 설정을 변화점 탐지 전에 해시로 굳히는 것이 핵심입니다.
# 1. **v3 검토:** 절차 설계(해시 동결, 불변식, 음성 대조)는 적절하다고 봤고, 사건 정의 부분은 동결하면 안 된다고 판단했습니다. 지적은 5개였습니다.
#    - 큰 사건 ±1년 대조의 변별력 부족
#    - 비중 돌파 사건이 결과 변수와 순환
#    - E1이 창간 연도와 코퍼스 진입 연도를 섞음
#    - 검증 게이트가 약함
#    - PREREG 누락 항목
# 2. **v4:** 위 5개를 반영하고, 시나리오별 우연 적중률을 먼저 출력해 주 시나리오를 규칙으로 정하게 했습니다.
# 3. **진단 결과:** 주 시나리오는 `NONE`입니다.
#    - S-A는 tol 0에서 0.52, ±1년에서 0.81로 어느 시나리오도 변별력이 없습니다.
#    - 그래서 사건 ±1년 대조는 민감도로 내리고, 연속 노출 시계열(비고정 저널 비중)을 주 분석으로 두기로 했습니다. 이 사양은 진단을 본 뒤 정했으므로 PREREG에 사후 명시로 적었습니다.
# 4. **v4.1:** 창간 전 논문 검토 파일, 제외 pid 처리(`F1X`), DOI 접두 보고서, 단계 스위치(`STAGE`), 고정 저널 14개 확정, 게이트 보완을 넣었습니다.
# 5. **현재 상태:**
#    - E1 충돌 저널은 AMAR, RailEng, T-ITS, TR-E, TRR 5개입니다.
#    - T-ITS 1997~1999 논문 9편은 오귀속 의심으로 검토 중이고, AMAR 2013년 6편은 온라인 선공개로 보입니다.
#    - 희소 DOI 접두 23편도 검토 중입니다.
#    - 검토 파일 세 개에 Crossref 정보를 붙인 단계까지 왔습니다.

# ## 앞으로 할 일
# **A. S0 마무리(지금 단계)**
# 1. 판정 보조 셀을 실행하고(Crossref 상태, 기준 게재지 비교), 세 `REVIEW_*.csv`의 `decision`을 직접 채웁니다. 기준은 "T-ITS 등 해당 저널에 실렸는가"입니다.
# 2. `diag`를 다시 실행해 제외 반영 후 충돌 목록이 줄었는지 확인합니다.
# 3. `events_catalog.csv`에 `source_url`, `retrieved_date`를 채웁니다. 게이트 대상은 S-B(high) 14건입니다.
# 4. `events_override.csv`를 작성합니다. 대상은 TRR, RailEng, TR-E, AMAR이고, T-ITS는 전부 `EXCLUDE`하면 필요 없습니다.
# 5. `variant_names` 7개를 실제 이름으로 교체합니다.
# 6. `STAGE="draft"`, `FORCE_PREREG=True`로 실행해 FAIL 0건을 확인하고, `PREREG.json`이 저장되면 `FORCE_PREREG`를 `False`로 되돌립니다.
# 7. `STAGE="freeze"`로 동결한 뒤 git commit과 tag를 남기고, PREREG.md에 "임계값과 `exposure_test`는 탐색 결과를 본 뒤 정했다"는 문장을 씁니다.
# **B. 이후 단계(계획서 기준)**
# - **S1:** 국가 층(OpenAlex, Lens)과 커버리지 감사. 창이 2010년 이후로 줄면 RQ4와 RQ5는 부록으로 내립니다.
# - **S2:** 측정 교정(가중치 4종, 위험군 재현율, 토픽 결측 성향 가중, K 민감도).
# - **S3~S4:** 토픽 증거 카드와 신뢰 등급, 양×질 4분면, 저널 구성 분해.
# - **S5:** 체제 식별(핵심). 사건 정보 없이 변화점을 탐지하고, 연속 노출 대조와 사건 시나리오 민감도를 적용합니다.
# - **S6~S9:** 체제별 갱신 분해, 지역·정렬(S1 통과 시), 후보 풀과 추출 규칙 동결.
# - **S10:** 투고 패키지(TRA 선례 검색, 규정 준수, AI 사용 선언).

# ## 주의할 점
# - 분석은 이후 모두 `F1X`(제외 pid 반영) 기준이어야 합니다. 원본 F 파일 해시는 바꾸지 않습니다.
# - 동결 이후 설정이나 임계값을 바꾸면 CHANGELOG와 multiverse에 모두 기록해야 합니다.
# - 연속 노출 주 분석(`exposure_test`)의 실제 구현은 S5에서 합니다. 지금 PREREG에는 사양만 있습니다.

In [ ]:
# ## 판독 결과

# **T-ITS 9편: 전부 `EXCLUDE`로 판정할 수 있습니다.**
# - DOI가 없는 8편은 화학, 식량, 일본 총리 연설, 교육 등 비교통 내용입니다. T-ITS(창간 2000)에 실린 논문이 아니라 오귀속입니다.
# - 나머지 1편("Emergency Vehicle Maneuvers…")은 Crossref상 T-ITS 게재가 맞지만 `cr_year`가 2002이고 F1 연도는 1998입니다. 정당한 1997~1999 T-ITS 논문이 아니라 연도 오류입니다.
# - 이 9편을 제외하면 T-ITS 코퍼스 첫 해는 2000이 되어 충돌이 자동으로 해소되므로 T-ITS의 override 행은 필요 없습니다.

# **AMAR 6편: `KEEP`.**
# - 6편 모두 게재지가 일치하고 `cr_year`가 2014(vol.1)입니다. F1의 2013은 온라인 선공개 연도로 보입니다.
# - AMAR가 큰 사건이나 S-B(high) 사건에 걸리지 않으면 게이트에는 영향이 없습니다. 기록용으로 override 행을 하나 두는 정도면 됩니다.

# **RAREPREFIX 23편은 세 부류로 나뉩니다.**
# - **명백한 오귀속(`EXCLUDE`) 7편:**
#   - AAP 6편(동물·의학·음성인지 등 비교통 주제, 접두 10.11648)
#   - JTTE 1편(Crossref상 도서 *Traffic and Granular Flow '17*의 챕터)
# - **교통 주제인데 DOI가 Crossref에 없는 15편:** CACIE 2, ETRR 1, JPT 1, TRIP 1, T-ITS 4, TRR 3, TransSci 4(Crossref 404)
#   - 다른 등록기관이나 잘못된 DOI일 수 있습니다. 저널 소속 자체는 맞을 가능성이 있어 제목 기반 조회가 필요합니다.
#   - 소속이 맞더라도 `doi_norm`이 틀린 것이므로 S1의 OpenAlex DOI 매칭에서 실패합니다. 이 15편은 DOI 오류 목록으로 따로 남기세요.
# - **판정 보류 1편:** AAP·JTTE 외에 위 어느 쪽에도 분명히 속하지 않는 행이 있으면 제목 조회 결과로 결정합니다.

# ## 실행할 셀

# **셀 1: 명백한 행 판정 기록**(검토 파일에 `decision`과 `reason`을 씁니다)

# ```python
# def load(n): return pd.read_csv(OUT / n, dtype=str, encoding="utf-8-sig").fillna("")
# def save(n, R): R.to_csv(OUT / n, index=False, encoding="utf-8-sig")

# R = load("REVIEW_T-ITS_prefounding.csv"); R["decision"] = "EXCLUDE"
# R["reason"] = np.where(R.cr_status == "200", "게재지는 T-ITS이나 F1 연도 1998≠Crossref 2002(연도 오류)", "DOI 없음·비교통 내용·창간(2000) 이전")
# save("REVIEW_T-ITS_prefounding.csv", R)

# R = load("REVIEW_AMAR_prefounding.csv"); R["decision"] = "KEEP"
# R["reason"] = "게재지 일치, Crossref 2014(vol.1), F1 2013은 online-first로 추정"
# save("REVIEW_AMAR_prefounding.csv", R)

# R = load("REVIEW_RAREPREFIX_prefounding.csv")
# m = R.venue_short.eq("AAP"); R.loc[m, "decision"] = "EXCLUDE"; R.loc[m, "reason"] = "비교통 주제(의학·축산 등), 희소 접두 10.11648, Crossref 404"
# m = R.venue_short.eq("JTTE") & R.cr_type.eq("book-chapter"); R.loc[m, "decision"] = "EXCLUDE"; R.loc[m, "reason"] = "Crossref 게재지가 도서(Traffic and Granular Flow '17)"
# save("REVIEW_RAREPREFIX_prefounding.csv", R)
# print(R.decision.replace("", "(미결)").value_counts())
# ```

# **셀 2: 남은 15편 제목 기반 Crossref 조회**(`ref`, `norm`, `requests`, `time`은 앞 셀의 것을 그대로 씁니다)

# ```python
# import difflib
# def tl(title):
#     try:
#         r = requests.get("https://api.crossref.org/works", timeout=30,
#                          params={"query.bibliographic": title, "rows": 3, "select": "DOI,title,container-title,issued", "mailto": "YOUR_EMAIL"})
#         it = r.json()["message"]["items"]
#         sim = lambda m: difflib.SequenceMatcher(None, norm(title), norm((m.get("title") or [""])[0])).ratio()
#         b = max(it, key=sim); dp = (b.get("issued", {}).get("date-parts") or [[None]])[0]
#         return dict(tl_doi=b.get("DOI", ""), tl_container=" | ".join(b.get("container-title", [])), tl_year=dp[0] if dp else "", tl_sim=round(sim(b), 2))
#     except Exception as e:
#         return dict(tl_doi="", tl_container=repr(e)[:60], tl_year="", tl_sim=0)

# R = load("REVIEW_RAREPREFIX_prefounding.csv"); todo = R.decision == ""
# X = []
# for t in R.loc[todo, "title"]: X.append(tl(t)); time.sleep(0.2)
# X = pd.DataFrame(X, index=R.index[todo])
# for c in X.columns: R.loc[todo, c] = X[c].astype(str)
# ok = todo & (R.tl_sim.replace("", "0").astype(float) >= 0.9) & (R.tl_container.map(norm) == R.venue_short.map(ref))
# R["suggest2"] = np.where(~todo, "", np.where(ok, "KEEP?(제목·게재지 일치, DOI만 오류)", "EXCLUDE?/REVIEW"))
# save("REVIEW_RAREPREFIX_prefounding.csv", R)
# print(R.loc[todo, ["venue_short", "year", "title", "tl_container", "tl_year", "tl_sim", "suggest2"]]
#       .assign(title=R.title.str[:45], tl_container=R.tl_container.str[:40]).to_string(index=False))
# ```

# - `KEEP?`는 제목이 0.9 이상 일치하고 게재지도 같은 경우입니다. 이때는 `KEEP`으로 하고 `reason`에 `tl_doi`(올바른 DOI)를 적으세요.
# - 일치하지 않거나 조회 실패(`REVIEW`)인 행은 제목과 `tl_container`를 눈으로 보고 판단하세요. 근거가 없으면 `EXCLUDE`하고 이유를 남기면 됩니다.

# **셀 3: 데이터 품질 점검**(S1/S2 항목이지만 이번 결과에 비추어 지금 한 번 확인할 가치가 있습니다)

# ```python
# # (a) T-ITS의 DOI 없는 논문(2000년 이후): 창간 전 9편과 같은 오귀속이 더 있는지
# t = F1X[(F1X.venue_short == "T-ITS") & F1X.doi_norm.isna()]
# print("T-ITS DOI 없음", len(t), "편, 연도별", t.year.value_counts().sort_index().to_dict())
# print(t[["pid", "year", "title"]].assign(title=t.title.str[:70]).head(25).to_string(index=False))

# # (b) F1 연도 vs Crossref 발행연도 불일치(고정 저널 표본 400편)
# s = F1X[F1X.venue_short.isin(PREREG["event"]["fixed_venues"]) & F1X.doi_norm.notna() & (F1X.year >= 2000)].sample(400, random_state=2026)
# cy = []
# for doi in s.doi_norm: cy.append(cr(doi).get("cr_year")); time.sleep(0.2)
# s = s.assign(cr_year=pd.to_numeric(pd.Series(cy, index=s.index), errors="coerce")); s["gap"] = (s.year - s.cr_year).abs()
# print("조회 성공", int(s.cr_year.notna().sum()), "| 불일치 ≥2년 비율", round(float((s.gap >= 2).sum() / s.cr_year.notna().sum()), 3))
# print(s[s.gap >= 2].groupby("venue_short").size().sort_values(ascending=False).head(10))
# ```

# - (a)에서 비교통 제목이 더 나오면 같은 `REVIEW_*_prefounding.csv` 방식으로 제외 목록을 확장해야 합니다. 이번 T-ITS 8편이 창간 전이라서만 잡혔을 가능성이 있습니다.
# - (b)에서 불일치 비율이 높으면 연도 오류가 시계열 분석 전반에 영향을 줍니다. 이 경우 변화점 연도의 신뢰도가 걸려 있으니 결과를 알려 주세요.

# ## 다음 순서
# 1. 셀 1과 셀 2를 실행하고 남은 `decision`을 모두 채웁니다.
# 2. 셀 3의 결과를 확인합니다.
# 3. `STAGE="diag"`를 다시 실행해 제외 pid 수와 충돌 목록이 줄었는지 봅니다.
# 4. 이후 `source_url`, override, `variant_names` 입력과 `draft` 단계는 이전 순서대로 진행합니다.

# 셀 2의 표와 셀 3의 출력을 붙여 주시면 남은 판정과 연도 오류 영향을 같이 보겠습니다.

In [8]:
def load(n): return pd.read_csv(OUT / n, dtype=str, encoding="utf-8-sig").fillna("")
def save(n, R): R.to_csv(OUT / n, index=False, encoding="utf-8-sig")

R = load("REVIEW_T-ITS_prefounding.csv"); R["decision"] = "EXCLUDE"
R["reason"] = np.where(R.cr_status == "200", "게재지는 T-ITS이나 F1 연도 1998≠Crossref 2002(연도 오류)", "DOI 없음·비교통 내용·창간(2000) 이전")
save("REVIEW_T-ITS_prefounding.csv", R)

R = load("REVIEW_AMAR_prefounding.csv"); R["decision"] = "KEEP"
R["reason"] = "게재지 일치, Crossref 2014(vol.1), F1 2013은 online-first로 추정"
save("REVIEW_AMAR_prefounding.csv", R)

R = load("REVIEW_RAREPREFIX_prefounding.csv")
m = R.venue_short.eq("AAP"); R.loc[m, "decision"] = "EXCLUDE"; R.loc[m, "reason"] = "비교통 주제(의학·축산 등), 희소 접두 10.11648, Crossref 404"
m = R.venue_short.eq("JTTE") & R.cr_type.eq("book-chapter"); R.loc[m, "decision"] = "EXCLUDE"; R.loc[m, "reason"] = "Crossref 게재지가 도서(Traffic and Granular Flow '17)"
save("REVIEW_RAREPREFIX_prefounding.csv", R)
print(R.decision.replace("", "(미결)").value_counts())

decision
(미결)       16
EXCLUDE     7
Name: count, dtype: int64


In [9]:
import difflib
def tl(title):
    try:
        r = requests.get("https://api.crossref.org/works", timeout=30,
                         params={"query.bibliographic": title, "rows": 3, "select": "DOI,title,container-title,issued", "mailto": "YOUR_EMAIL"})
        it = r.json()["message"]["items"]
        sim = lambda m: difflib.SequenceMatcher(None, norm(title), norm((m.get("title") or [""])[0])).ratio()
        b = max(it, key=sim); dp = (b.get("issued", {}).get("date-parts") or [[None]])[0]
        return dict(tl_doi=b.get("DOI", ""), tl_container=" | ".join(b.get("container-title", [])), tl_year=dp[0] if dp else "", tl_sim=round(sim(b), 2))
    except Exception as e:
        return dict(tl_doi="", tl_container=repr(e)[:60], tl_year="", tl_sim=0)

R = load("REVIEW_RAREPREFIX_prefounding.csv"); todo = R.decision == ""
X = []
for t in R.loc[todo, "title"]: X.append(tl(t)); time.sleep(0.2)
X = pd.DataFrame(X, index=R.index[todo])
for c in X.columns: R.loc[todo, c] = X[c].astype(str)
ok = todo & (R.tl_sim.replace("", "0").astype(float) >= 0.9) & (R.tl_container.map(norm) == R.venue_short.map(ref))
R["suggest2"] = np.where(~todo, "", np.where(ok, "KEEP?(제목·게재지 일치, DOI만 오류)", "EXCLUDE?/REVIEW"))
save("REVIEW_RAREPREFIX_prefounding.csv", R)
print(R.loc[todo, ["venue_short", "year", "title", "tl_container", "tl_year", "tl_sim", "suggest2"]]
      .assign(title=R.title.str[:45], tl_container=R.tl_container.str[:40]).to_string(index=False))

venue_short year                                         title                             tl_container tl_year tl_sim                  suggest2
      CACIE 2016 Semantic Enrichment for Building Information  Computer-Aided Civil and Infrastructure     2016    1.0 KEEP?(제목·게재지 일치, DOI만 오류)
      CACIE 2017 Mountain Railway Alignment Optimization with  Computer-Aided Civil and Infrastructure     2017    1.0 KEEP?(제목·게재지 일치, DOI만 오류)
       ETRR 2025 Improving urban delivery with extended cargo        European Transport Research Review    2025    1.0 KEEP?(제목·게재지 일치, DOI만 오류)
        JPT 2025 Integrating demand-responsive services into p         Journal of Public Transportation    2025    1.0 KEEP?(제목·게재지 일치, DOI만 오류)
      T-ITS 2018 A New CNN-Based Method for Multi-Directional  IEEE Transactions on Intelligent Transpo    2018    1.0 KEEP?(제목·게재지 일치, DOI만 오류)
      T-ITS 2018 Self-localization based on visual lane markin IEEE Transactions on Intelligent Transpo    2018   0.66           E

In [10]:
# (a) T-ITS의 DOI 없는 논문(2000년 이후): 창간 전 9편과 같은 오귀속이 더 있는지
t = F1X[(F1X.venue_short == "T-ITS") & F1X.doi_norm.isna()]
print("T-ITS DOI 없음", len(t), "편, 연도별", t.year.value_counts().sort_index().to_dict())
print(t[["pid", "year", "title"]].assign(title=t.title.str[:70]).head(25).to_string(index=False))

# (b) F1 연도 vs Crossref 발행연도 불일치(고정 저널 표본 400편)
s = F1X[F1X.venue_short.isin(PREREG["event"]["fixed_venues"]) & F1X.doi_norm.notna() & (F1X.year >= 2000)].sample(400, random_state=2026)
cy = []
for doi in s.doi_norm: cy.append(cr(doi).get("cr_year")); time.sleep(0.2)
s = s.assign(cr_year=pd.to_numeric(pd.Series(cy, index=s.index), errors="coerce")); s["gap"] = (s.year - s.cr_year).abs()
print("조회 성공", int(s.cr_year.notna().sum()), "| 불일치 ≥2년 비율", round(float((s.gap >= 2).sum() / s.cr_year.notna().sum()), 3))
print(s[s.gap >= 2].groupby("venue_short").size().sort_values(ascending=False).head(10))

T-ITS DOI 없음 93 편, 연도별 {1997: 3, 1998: 1, 1999: 4, 2000: 3, 2001: 4, 2002: 3, 2003: 4, 2004: 5, 2005: 1, 2006: 2, 2007: 12, 2008: 14, 2009: 1, 2010: 7, 2011: 3, 2012: 5, 2013: 1, 2016: 2, 2017: 16, 2018: 2}
  pid  year                                                                  title
59676  2018                      Cooperative Vehicular Networking: A Survey | NIST
59677  2018 Roadside Magnetic Sensor System for Vehicle Detection in Urban Environ
63670  2017          パラメータ自己適応フレームワークを通した二次元密深さマップへの三次元LiDAR点群の変換【Powered by NICT】
63672  2017                        ノードベースボトルネック予測とV2X通信による増加した交通流【Powered by NICT】
63674  2017      M~2bmtによる車線検出システムのための自動グランドトルース生成と検証の強化されたアルゴリズム【Powered by NICT】
63678  2017                     車車間とV2I通信を用いた速度予測に基づく実時間エネルギー管理戦略【Powered by NICT】
63679  2017      3次元輸送GISデータベースを用いたマルチレベル道路網における車両ナビゲーションのための角差分法【Powered by NICT】
63688  2017                       Wi‐Fi位置決めシステムのための無線マップ構築手法の性能評価【Powered by NICT】
63689  2017               都市環境におけるMultirequest

KeyboardInterrupt: 

In [11]:
cjk_re  = r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]"
junk_re = r"Powered by NICT|Picture of the Cover|Front Cover|Table of Contents|Editorial Board"
d = F1.assign(nodoi=F1.doi_norm.isna(),
              r_cjk=F1.title.str.contains(cjk_re, regex=True, na=False),
              r_junk=F1.title.str.contains(junk_re, regex=True, na=False),
              r_sub=F1.title.str.contains(r":- .* -$", regex=True, na=False) & F1.doi_norm.isna())
cand = d[d.r_cjk | d.r_junk | d.r_sub]
print(cand.groupby("venue_short")[["r_cjk", "r_junk", "r_sub"]].sum())
print(cand.topic150.value_counts().head(10))          # 어느 토픽에 들어갔나
print(cand[~cand.venue_short.eq("T-ITS")][["venue_short", "year", "title"]].head(30).to_string(index=False))

             r_cjk  r_junk  r_sub
venue_short                      
AAP             61      57      0
ITS Mag          4       2      0
JTG              4       0      0
T-ITS           21      17      6
T-IV             7       4      0
T-TE            18      13      0
TR-B             3       0      0
TR-C             2       0      0
TR-D             4       0      0
TR-E             2       0      0
TRR             71       0      0
TranspRev        1       0      0
VT Mag          10       8      0
topic150
49.0     37
132.0     3
28.0      1
38.0      1
125.0     1
76.0      1
135.0     1
Name: count, dtype: int64
venue_short  year                                                                                            title
    ITS Mag  2021                                                              道路車両のための交通管制システム技術:特許分析【JST・京大機械翻訳】
       TR-E  2021                                                 オンラインB2B輸送マッチングプラットフォームのための勝者決定問題の解決【JST・京大機械翻訳】
       T-TE  2021         

In [12]:
ex = cand.assign(rule=np.select([cand.r_cjk, cand.r_junk], ["CJK 제목", "표지·NICT 등 비연구 문구"], "제목:- 부제 - 형식 + DOI 없음"))
ex[["pid", "venue_short", "year", "title", "doi_norm", "rule"]].to_csv(OUT / "exclude_pids.csv", index=False, encoding="utf-8-sig")
print(len(ex), "편 저장")

215 편 저장


In [13]:
cjk_re  = r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]"
tag_re  = r"【(?:JST・京大機械翻訳|Powered by NICT)】"
d = F1.assign(r_cjk=F1.title.str.contains(cjk_re, regex=True, na=False),
              r_tag=F1.title.str.contains(tag_re, regex=True, na=False))

# (1) CJK인데 번역 꼬리표가 없는 행: 정당한 논문일 수 있음
odd = d[d.r_cjk & ~d.r_tag]
print("CJK이나 꼬리표 없음:", len(odd), "편")
print(odd.groupby("venue_short").size())
print(odd[["venue_short", "year", "title", "doi_norm"]].assign(title=odd.title.str[:70]).head(40).to_string(index=False))

# (2) 토픽 49: 후보 밖에 일본어·번역 제목이 남았나
t49 = d[d.topic150 == 49]
print("토픽49 전체", len(t49), "| 꼬리표 포함", int(t49.r_tag.sum()), "| CJK 포함", int(t49.r_cjk.sum()))
print(t49[["venue_short", "year", "title"]].assign(title=t49.title.str[:60]).sample(min(20, len(t49)), random_state=2026).to_string(index=False))

# (3) 꼬리표 행에 대응하는 영문 원본이 코퍼스에 있는지: 같은 저널·연도에서 번역 제목의 영문 부분 일치는 어려우므로, 연도별 건수만 본다
print(d[d.r_tag].groupby("year").size().to_string())

CJK이나 꼬리표 없음: 82 편
venue_short
AAP           1
T-ITS         5
TR-B          3
TR-C          1
TRR          71
TranspRev     1
dtype: int64
venue_short  year                                                                  title doi_norm
        TRR  2016                          起点,終点データに着目した米国・カナダ間2大国境検問所を通過した国際トラック移動特性の確性      NaN
        TRR  2016                               大型貨物移動の燃料電池牽引における天然ガスの実行可能性 経済,効率および技術因子      NaN
        AAP  2016                                            標準化した現場飲酒運転検査(SFST)と認識機能の尺度      NaN
      T-ITS  2016            MEMS IMUと単一周波グローバルポジショニングシステム(GPS)受信機を用いた都市環境での高精度車両ナビゲーション      NaN
        TRR  2015               車両アドホック情報網から安全運転を警告する際に適切な状況判定をするための観察‐運転者‐学習の手法を原理とする規範      NaN
        TRR  2015                1993~2013年の米国民間航空ショーでの墜落事故 負荷,致命的なリスク要因及び航空機墜落のリスク指標の評価      NaN
        TRR  2014                                   輸送費解析 米国の東部湾岸沿岸内水路,アーカンソー川,およびレッドリバー      NaN
        TRR  2014                                Connected Vehicle Technolog

In [16]:
junk_re = r"Picture of the Cover|Front Cover|Table of Contents|Editorial Board"
d["r_junk"] = d.title.str.contains(junk_re, regex=True, na=False)
d["r_sub"]  = d.title.str.contains(r":- .* -$", regex=True, na=False) & d.doi_norm.isna()

ex = d[d.r_tag | d.r_junk | d.r_sub].copy()
ex["rule"] = np.select([ex.r_tag, ex.r_junk], ["JST·NICT 번역 레코드", "표지·목차 등 비연구"], "제목:- 부제 - 형식 + DOI 없음")
ex[["pid", "venue_short", "year", "title", "doi_norm", "rule"]].to_csv(OUT / "exclude_pids.csv", index=False, encoding="utf-8-sig")
print(len(ex), "편 저장;", ex.rule.value_counts().to_dict())

# 꼬리표 없는 CJK는 눈 확인용 검토 파일(g6 게이트 때문에 decision을 반드시 채워야 함)
rv = d[d.r_cjk & ~d.r_tag & ~d.r_junk & ~d.r_sub][["pid", "venue_short", "year", "title", "doi_norm"]].copy()
rv["decision"] = ""; rv["reason"] = ""
rv.to_csv(OUT / "REVIEW_CJKNOTAG_prefounding.csv", index=False, encoding="utf-8-sig")
print("검토 대상", len(rv), "편")

133 편 저장; {'JST·NICT 번역 레코드': 126, '제목:- 부제 - 형식 + DOI 없음': 6, '표지·목차 등 비연구': 1}
검토 대상 82 편


In [17]:
cjk_re = r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]"
tag_re = r"【(?:JST・京大機械翻訳|Powered by NICT)】"
d = F1.assign(r_cjk=F1.title.str.contains(cjk_re, regex=True, na=False),
              r_tag=F1.title.str.contains(tag_re, regex=True, na=False),
              r_jg=F1.title.str.contains(r"J-GLOBAL", na=False),
              r_junk=F1.title.str.contains(r"Picture of the Cover|Front Cover|Table of Contents|Editorial Board|^\[Conference Calendar\]|^Bibliographic Section", regex=True, na=False),
              r_sub=F1.title.str.contains(r":- .* -$", regex=True, na=False) & F1.doi_norm.isna(),
              nodoi=F1.doi_norm.isna())
d["r_cjk_nodoi"] = d.r_cjk & d.nodoi
hit = d.r_tag | d.r_jg | d.r_cjk_nodoi | d.r_junk | d.r_sub
ex = d[hit].copy()
ex["rule"] = np.select([ex.r_tag, ex.r_jg, ex.r_cjk_nodoi, ex.r_junk],
                       ["JST·NICT 번역 레코드(꼬리표)", "J-GLOBAL 레코드", "CJK 제목 + DOI 없음", "표지·목차·칼럼 등 비연구"],
                       "제목:- 부제 - 형식 + DOI 없음")
ex[["pid","venue_short","year","title","doi_norm","rule"]].to_csv(OUT/"exclude_pids.csv", index=False, encoding="utf-8-sig")
print(len(ex), ex.rule.value_counts().to_dict())

# CJK인데 DOI가 있고 꼬리표도 없는 행(정당할 수 있음): 제외하지 않고 개수만 확인
odd = d[d.r_cjk & ~d.nodoi & ~d.r_tag]
print("CJK이나 DOI 있음:", len(odd))
if len(odd): print(odd[["venue_short", "year", "title", "doi_norm"]].assign(title=odd.title.str[:60]).head(20).to_string(index=False))

# 검토 파일은 규칙이 흡수했으므로 삭제(남기면 g6 게이트가 빈 decision에 걸림)
(OUT/"REVIEW_CJKNOTAG_prefounding.csv").unlink(missing_ok=True)

251 {'JST·NICT 번역 레코드(꼬리표)': 126, 'CJK 제목 + DOI 없음': 70, '표지·목차·칼럼 등 비연구': 37, 'J-GLOBAL 레코드': 12, '제목:- 부제 - 형식 + DOI 없음': 6}
CJK이나 DOI 있음: 0


In [19]:
j = ex[ex.rule == "표지·목차·칼럼 등 비연구"]
print(j.groupby("venue_short").size().sort_values(ascending=False).to_string())
print(j.title.str[:50].value_counts().head(15).to_string())

venue_short
TransSci    35
ITS Mag      1
T-ITS        1
title
Bibliographic Section                     35
[Conference Calendar]                      1
Picture of the Cover:Toward the Future     1


In [20]:
# (a) RAREPREFIX 나머지 16편 판정
R = load("REVIEW_RAREPREFIX_prefounding.csv")
sim = pd.to_numeric(R.tl_sim.replace("", np.nan), errors="coerce")
yr_ok = R.tl_year.astype(str).str.replace(r"\.0$", "", regex=True) == R.year
ven_ok = R.tl_container.map(norm) == R.venue_short.map(ref)
todo = R.decision == ""
keep = todo & ven_ok & yr_ok & (sim >= 0.6)
R.loc[keep, "decision"] = "KEEP"; R.loc[keep, "reason"] = "제목 조회로 게재지·연도 일치, DOI만 오류. 올바른 DOI=" + R.tl_doi[keep]
rest = (R.decision == "")
R.loc[rest, "decision"] = "EXCLUDE"; R.loc[rest, "reason"] = "제목 조회로 게재지·연도 검증 불가(TRR 1977~91, 분석 창 밖)"
save("REVIEW_RAREPREFIX_prefounding.csv", R)
R[keep][["pid","doi_norm","tl_doi"]].rename(columns={"doi_norm":"doi_wrong","tl_doi":"doi_fixed"}) \
    .to_csv(OUT/"doi_fixes.csv", index=False, encoding="utf-8-sig")
print(R.decision.value_counts().to_dict())

# (b) 세 검토 파일 모두 미결 0인지
for f in sorted(OUT.glob("REVIEW_*_prefounding.csv")):
    x = pd.read_csv(f, dtype=str, encoding="utf-8-sig").fillna("")
    print(f.name, "미결", int((~x.decision.isin(["EXCLUDE","KEEP"])).sum()))

# (c) CHANGELOG (결과 열람 전 정리임을 명시)
for line in [
 f"- {now()} | 데이터 오염 정리: JST·NICT 꼬리표 126, CJK+DOI없음 70, 표지·서지섹션·칼럼 37, J-GLOBAL 12, 부제형식+DOI없음 6 = 251편 제외(exclude_pids.csv) | 변화점·체제 결과 열람 전에 오염을 발견해 정리, 규칙 정규식은 데이터를 본 뒤 확정 | 모든 파생 분석(F1X)",
 f"- {now()} | 창간 전 논문 검토: T-ITS 9편 EXCLUDE, AMAR 6편 KEEP(online-first), RAREPREFIX 판정 | 게재지 오귀속·연도 오류 | 사건 E1",
 f"- {now()} | 제목 조회 KEEP 기준 유사도 0.9 -> 0.6(게재지·연도 일치 조건 병행) | 부제 절단으로 유사도 저하 | RAREPREFIX 판정",
 f"- {now()} | doi_fixes.csv: DOI 오류 논문 목록 보관(S1 OpenAlex 조회 시 doi_wrong -> doi_fixed 치환) | Crossref 404 | S1 국가 층",
]: changelog(line)

{'KEEP': 13, 'EXCLUDE': 10}
REVIEW_AMAR_prefounding.csv 미결 0
REVIEW_RAREPREFIX_prefounding.csv 미결 0
REVIEW_T-ITS_prefounding.csv 미결 0


In [22]:
   C = pd.read_csv(OUT/"events_catalog.csv")
   print(C[(C.confidence=="high") & C.event_type.isin(["founding","founding_oa","split","rename","rename_publisher"])][["venue_short","event_type","event_year"]].to_string(index=False))

venue_short       event_type  event_year
   TransSci         founding        1967
        TRR         founding        1963
        TRR           rename        1974
        TRR           rename        2000
        AAP         founding        1969
       TR-A            split        1979
       TR-B            split        1979
  TranspRev         founding        1981
      CACIE         founding        1986
      CACIE rename_publisher        1998
        JTG         founding        1993
  TranspPol         founding        1993
       TR-C         founding        1993
        JPT      founding_oa        1996
       TR-D         founding        1996
       TR-E           rename        1997
       TR-F         founding        1998
      T-ITS         founding        2000
       ETRR         founding        2008
       RTBM         founding        2011
      IJTST         founding        2012
       JTTE         founding        2014
       T-TE         founding        2015
       T-IV     

In [23]:
C = pd.read_csv(OUT/"events_catalog.csv")
for c in ("source_url","retrieved_date"):
    if c not in C.columns: C[c] = ""
g = C[(C.confidence=="high") & C.event_type.isin(["founding","founding_oa","split","rename","rename_publisher"]) & (C.event_year>=2000)]
g[["venue_short","event_type","event_year","source_url","retrieved_date"]].to_csv(OUT/"events_catalog_FILL.csv", index=False, encoding="utf-8-sig")
print(len(g), "건 -> events_catalog_FILL.csv에 URL과 날짜를 채우세요")

14 건 -> events_catalog_FILL.csv에 URL과 날짜를 채우세요


In [25]:
C = pd.read_csv(OUT/"events_catalog.csv")
F = pd.read_csv(OUT/"events_catalog_FILL.csv")
for c in ("source_url","retrieved_date"):
    if c not in C.columns: C[c] = ""
k = ["venue_short","event_type","event_year"]
C = C.drop(columns=["source_url","retrieved_date"]).merge(F[k+["source_url","retrieved_date"]], on=k, how="left")
C[["source_url","retrieved_date"]] = C[["source_url","retrieved_date"]].fillna("")
C.to_csv(OUT/"events_catalog.csv", index=False, encoding="utf-8-sig")
print(C[C.event_year>=2000][["venue_short","event_type","event_year","source_url"]].assign(source_url=C.source_url.str[:40]).to_string(index=False))

venue_short       event_type  event_year                               source_url
        TRR           rename        2000 https://trb.org/Publications/PubsTRRJour
        TRR publisher_change        2018                                         
        AAP publisher_change        2005                                         
       TR-B    editor_change        2003                                         
  TranspRev frequency_change        2004                                         
        JPT publisher_change        2022                                         
       TR-E    editor_change        2013                                         
       TR-E volume_expansion        2024                                         
      T-ITS         founding        2000 https://consulta-ccn.prd.ibict.br/vufind
      T-ITS       parent_org        2005                                         
     VT Mag         founding        2006                                         
       IJST     

In [26]:
C = pd.read_csv(OUT/"events_catalog.csv")
g = C[(C.event_year>=2000) & C.event_type.isin(["founding","founding_oa","split","rename","rename_publisher"])]
print(g.groupby("confidence").size())
print(g[(g.confidence=="high") & (g.source_url.fillna("").str.len()==0)][["venue_short","event_type","event_year","confidence"]])

confidence
high      14
medium     7
dtype: int64
Empty DataFrame
Columns: [venue_short, event_type, event_year, confidence]
Index: []


In [27]:
   O = pd.read_csv(OUT/"events_override_RailEng.csv", encoding="utf-8-sig")
   p = OUT/"events_override.csv"
   if p.exists(): O = pd.concat([pd.read_csv(p, encoding="utf-8-sig"), O]).drop_duplicates("venue_short", keep="last")
   O.to_csv(p, index=False, encoding="utf-8-sig"); print(O)

  venue_short  entry_year_ruling                                       source_url                                               note
0     RailEng               2020  https://portal.issn.org/resource/ISSN/2662-4753  Journal of Modern Transportation(2011-2019)를 이...


In [28]:
STAGE = "diag"
CHECKS.clear(); STATE.clear()
safe("sec0", sec0)


[0] 진단: 오귀속 검토 -> E1 충돌 -> 시나리오별 우연 적중률 -> 주 시나리오 결정
제외 pid 269편 반영(F1 111,685 -> F1X 111,416)
연속 노출 시계열 저장 (59개 연도)

(a) 창간 연도 vs 코퍼스 첫 논문 연도(제외 반영 후) 불일치 저널
 - AMAR: 창간 2014 / 코퍼스 2013 / 판정 없음(pending) -> 코퍼스가 창간보다 앞섬(검토 파일 확인)
 - RailEng: 창간 1993 / 코퍼스 2020 / 판정 2020 -> 코퍼스가 창간보다 늦음(범위 편입 시점)
 - TR-E: 창간 1972 / 코퍼스 1997 / 판정 없음(pending) -> 코퍼스가 창간보다 늦음(범위 편입 시점)
 - TRR: 창간 1963 / 코퍼스 1967 / 판정 없음(pending) -> 코퍼스가 창간보다 늦음(범위 편입 시점)

    검토 파일 판정 현황(저널별):
decision     EXCLUDE  KEEP
venue_short               
AAP                6     0
AMAR               0     6
CACIE              0     2
ETRR               0     1
JPT                0     1
JTTE               1     0
T-ITS              9     4
TRIP               0     1
TRR                3     0
TransSci           0     4
    pending ['AMAR', 'TR-E', 'TRR'] -> events_override_TEMPLATE.csv 저장(entry_year_ruling=코퍼스 진입 사건으로 쓸 연도, source_url 필수)

(b) 2000~2025 매년 논문이 있는 저널(F1X 기준) 14개: ['AAP', 'CACIE', 'JTG', 'T-ITS', 'TR-A', 'TR-B', 'T

In [29]:
p = OUT/"events_override.csv"
O = pd.read_csv(p, encoding="utf-8-sig")
new = pd.DataFrame([dict(venue_short="TRR", entry_year_ruling=1967,
    source_url="https://trb.org/Publications/PubsTRRJournal.aspx?em=1",
    note="Highway Research Record로 1963 창간(1974 개명, 2000 부제 추가). 코퍼스는 1967부터 수록. 2000년 이전이라 사건 목록에는 영향 없음")])
O = pd.concat([O, new]).drop_duplicates("venue_short", keep="last")
O.to_csv(p, index=False, encoding="utf-8-sig"); print(O[["venue_short","entry_year_ruling"]])
changelog(f"- {now()} | events_override: RailEng 2020, TRR 1967(코퍼스 진입 판정, source_url 포함) | 게이트 g3 대응 | 사건 목록 불변")

  venue_short  entry_year_ruling
0     RailEng               2020
0         TRR               1967


In [30]:
PREREG["risk_name"]["variant_names"] = ["...", "...", "...", "...", "...", "...", "..."]   # 실제 이름 7개
STAGE = "draft"; FORCE_PREREG = True
CHECKS.clear(); STATE.clear()
for n, f in (("sec0", sec0), ("sec1", sec1), ("sec2", sec2), ("sec3", sec3), ("sec4", sec4), ("sec5", sec5), ("sec6", sec6)): safe(n, f)
LOG = pd.DataFrame(CHECKS); LOG.index += 1
print(LOG[["검사", "상태", "관측"]].to_string())


[0] 진단: 오귀속 검토 -> E1 충돌 -> 시나리오별 우연 적중률 -> 주 시나리오 결정
제외 pid 269편 반영(F1 111,685 -> F1X 111,416)
연속 노출 시계열 저장 (59개 연도)

(a) 창간 연도 vs 코퍼스 첫 논문 연도(제외 반영 후) 불일치 저널
 - AMAR: 창간 2014 / 코퍼스 2013 / 판정 없음(pending) -> 코퍼스가 창간보다 앞섬(검토 파일 확인)
 - RailEng: 창간 1993 / 코퍼스 2020 / 판정 2020 -> 코퍼스가 창간보다 늦음(범위 편입 시점)
 - TR-E: 창간 1972 / 코퍼스 1997 / 판정 없음(pending) -> 코퍼스가 창간보다 늦음(범위 편입 시점)
 - TRR: 창간 1963 / 코퍼스 1967 / 판정 1967 -> 코퍼스가 창간보다 늦음(범위 편입 시점)

    검토 파일 판정 현황(저널별):
decision     EXCLUDE  KEEP
venue_short               
AAP                6     0
AMAR               0     6
CACIE              0     2
ETRR               0     1
JPT                0     1
JTTE               1     0
T-ITS              9     4
TRIP               0     1
TRR                3     0
TransSci           0     4
    pending ['AMAR', 'TR-E'] -> events_override_TEMPLATE.csv 저장(entry_year_ruling=코퍼스 진입 사건으로 쓸 연도, source_url 필수)

(b) 2000~2025 매년 논문이 있는 저널(F1X 기준) 14개: ['AAP', 'CACIE', 'JTG', 'T-ITS', 'TR-A', 'TR-B', 'TR-C', 'TR-D', 

In [ ]:
# ## 판독

# **31개 검사가 모두 PASS이고 FAIL과 SKIP은 없습니다.** 게이트 g1~g7이 전부 통과했고 `PREREG.json`도 저장됐으니 동결할 수 있습니다.

# - **g3 통과:** TRR override(1967)가 효과를 냈습니다. 남은 pending인 AMAR와 TR-E는 게이트 대상 사건이나 큰 사건에 걸리지 않아 막지 않습니다. 동결 파일에는 `pending=True`로 그대로 기록됩니다. 두 저널 모두 주 시나리오와 무관해서 이대로 둬도 됩니다.
# - **큰 사건은 6개**입니다(T-ITS 2011, TR-C 2013, TR-D 2016, TR-F 2018, T-TE 2021, T-IV 2023). 그중 진입 사건은 3개입니다. 비중 돌파 사건은 결과 변수와 순환하기 때문에 민감도 시나리오(S-E)로만 쓰입니다.
# - **음성 대조는 설계대로 작동했습니다.** 라벨을 섞으면 저자 주제 집중도가 0.344에서 0.179로 떨어지고, 집계 분포는 보존됩니다.
# - **외부 대조의 0.999는 독립 검증이 아닙니다.** F1의 `cited_by_count`가 OpenAlex에서 왔다면 같은 출처끼리의 일치라 거의 1이 나옵니다. 데이터 처리 오류가 없다는 정도의 의미이고, 타당성 검증으로는 쓰지 마세요. 보고서에는 "수집 일관성 점검"이라고 쓰는 편이 정직합니다.

# ## 동결 전에 챙길 것

# 1. **`variant_names`를 스크립트 파일에도 넣으세요.** 지금 PREREG.json은 노트북 메모리에서 바꾼 이름으로 저장됐습니다. 커널을 재시작하면 파일의 `TODO_` 값이 돌아와 PREREG 내용이 달라지고, `FORCE_PREREG=False` 상태에서는 "PREREG 변경 없음" 검사가 FAIL합니다.
# 2. **CHANGELOG에 v4 근거를 직접 적으세요.** `sec1`의 긴 변경 사유는 기존 PREREG.json이 있을 때만 기록됩니다. 이번에는 처음 저장이라 기록되지 않았습니다. 임계값과 `exposure_test`가 탐색 결과를 본 뒤 정해졌다는 사후 명시는 여기에 꼭 있어야 합니다.

# ```python
# changelog(f"- {now()} | PREREG v4 저장(eb2f284f) | (1) 큰 사건 규칙을 계획서의 '증가폭 상위 50%'에서 '5% 연속 2년 돌파'로 변경, (2) 진단에서 모든 사건 시나리오의 우연 적중률이 기준(0.50) 초과 -> 주 시나리오 NONE, 연속 노출 exposure_test를 진단 결과를 본 뒤 구체화(사후 명시), 사건 시나리오는 민감도, (3) E1을 창간/코퍼스 진입으로 분리, (4) 게이트 강화, (5) fixed_venues 14개, (6) 오염 레코드 269편 제외와 창간 전 논문 검토 | 변화점·체제 결과 열람 전 | 사건 대조(S5.6), 노출 분석, 모든 파생 분석(F1X)")
# ```

# ## 동결 실행

# ```python
# FORCE_PREREG = False
# STAGE = "freeze"
# CHECKS.clear(); STATE.clear()
# for n, f in (("sec0", sec0), ("sec1", sec1), ("sec2", sec2), ("sec3", sec3), ("sec4", sec4), ("sec5", sec5), ("sec6", sec6)): safe(n, f)
# LOG = pd.DataFrame(CHECKS); LOG.index += 1
# print(LOG[["검사", "상태"]].to_string()); print(LOG.상태.value_counts().to_dict())
# ```

# 마지막 줄이 `{'PASS': 32}` 안팎(동결 검사 1건 추가)이어야 합니다. FAIL이 나오면 붙여 주세요.

# ## 동결 직후 (같은 날)

# ```bash
# cd ~/Research/Minnesota_Transport
# git add paper1/00_S0 <S0 스크립트 경로>
# git commit -m "S0 frozen: PREREG v4, events, exclusions"
# git tag -a s0-frozen -m "S0 freeze $(date -I)"
# git push origin main --tags   # 원격이 있으면. 없으면 OSF 등록
# ```

# `doi_fixes.csv`, `exclude_pids.csv`, 검토 파일도 `00_S0` 안에 있어서 함께 커밋됩니다. 코드 내부의 해시와 타임스탬프는 스스로 쓴 기록이라, 독립 시각 증거는 git 원격이나 OSF에서 나옵니다.

# ## 마지막으로 PREREG.md

# 앞서 정리한 문장(임계값은 탐색 후 확정, 오염 정리는 결과 열람 전, 2020년 이후 구간은 이미 봄, 논문 2는 논문 1의 결과 변수를 쓰지 않음)에 위의 외부 대조 한계 한 줄을 더하면 됩니다. 초안이 필요하면 CHANGELOG 내용 그대로 정리해 드리겠습니다.

# 동결이 끝나면 다음은 S1(국가 층과 커버리지 감사)입니다. 이때 `doi_fixes.csv`의 15편을 OpenAlex 조회에 반영하는 것을 잊지 마세요.

In [31]:
changelog(f"- {now()} | PREREG v4 저장(eb2f284f) | (1) 큰 사건 규칙을 계획서의 '증가폭 상위 50%'에서 '5% 연속 2년 돌파'로 변경, (2) 진단에서 모든 사건 시나리오의 우연 적중률이 기준(0.50) 초과 -> 주 시나리오 NONE, 연속 노출 exposure_test를 진단 결과를 본 뒤 구체화(사후 명시), 사건 시나리오는 민감도, (3) E1을 창간/코퍼스 진입으로 분리, (4) 게이트 강화, (5) fixed_venues 14개, (6) 오염 레코드 269편 제외와 창간 전 논문 검토 | 변화점·체제 결과 열람 전 | 사건 대조(S5.6), 노출 분석, 모든 파생 분석(F1X)")

In [32]:
FORCE_PREREG = False
STAGE = "freeze"
CHECKS.clear(); STATE.clear()
for n, f in (("sec0", sec0), ("sec1", sec1), ("sec2", sec2), ("sec3", sec3), ("sec4", sec4), ("sec5", sec5), ("sec6", sec6)): safe(n, f)
LOG = pd.DataFrame(CHECKS); LOG.index += 1
print(LOG[["검사", "상태"]].to_string()); print(LOG.상태.value_counts().to_dict())


[0] 진단: 오귀속 검토 -> E1 충돌 -> 시나리오별 우연 적중률 -> 주 시나리오 결정
제외 pid 269편 반영(F1 111,685 -> F1X 111,416)
연속 노출 시계열 저장 (59개 연도)

(a) 창간 연도 vs 코퍼스 첫 논문 연도(제외 반영 후) 불일치 저널
 - AMAR: 창간 2014 / 코퍼스 2013 / 판정 없음(pending) -> 코퍼스가 창간보다 앞섬(검토 파일 확인)
 - RailEng: 창간 1993 / 코퍼스 2020 / 판정 2020 -> 코퍼스가 창간보다 늦음(범위 편입 시점)
 - TR-E: 창간 1972 / 코퍼스 1997 / 판정 없음(pending) -> 코퍼스가 창간보다 늦음(범위 편입 시점)
 - TRR: 창간 1963 / 코퍼스 1967 / 판정 1967 -> 코퍼스가 창간보다 늦음(범위 편입 시점)

    검토 파일 판정 현황(저널별):
decision     EXCLUDE  KEEP
venue_short               
AAP                6     0
AMAR               0     6
CACIE              0     2
ETRR               0     1
JPT                0     1
JTTE               1     0
T-ITS              9     4
TRIP               0     1
TRR                3     0
TransSci           0     4
    pending ['AMAR', 'TR-E'] -> events_override_TEMPLATE.csv 저장(entry_year_ruling=코퍼스 진입 사건으로 쓸 연도, source_url 필수)

(b) 2000~2025 매년 논문이 있는 저널(F1X 기준) 14개: ['AAP', 'CACIE', 'JTG', 'T-ITS', 'TR-A', 'TR-B', 'TR-C', 'TR-D', 

In [33]:
#

In [ ]:
# ## 판독
# **S0 동결이 끝났습니다.** 32개 검사가 모두 PASS이고(`{'PASS': 32}`), `사건 목록 동결`도 PASS입니다. `FREEZE_LOG.csv`에 해시 9개가 기록됐습니다(`EVENTS_FROZEN.csv` cf91c021…, 제외 pid, 검토 파일 3개, override, 카탈로그, 시나리오 표, 주 시나리오).
# 남은 건 **독립 타임스탬프**뿐입니다. 로그의 시각은 스스로 쓴 기록이고, `git_commit`이 `None`으로 찍혀 있습니다. 지금 커밋하세요.
# ```bash
# cd ~/Research/Minnesota_Transport
# git status --short | head        # paper1/00_S0 안의 파일이 보이는지 확인
# git add paper1/00_S0 <S0 스크립트 경로>
# git commit -m "S0 frozen: PREREG v4 (eb2f284f), events, exclusions"
# git tag -a s0-frozen -m "S0 freeze 2026-10-06"
# git push origin main --tags      # 원격이 없으면 OSF 등록으로 대체
# ```

# ## 같이 끝내야 할 3가지
# 1. **`variant_names` 7개의 실제 이름을 확인하세요.** 이번 PREREG가 올바른 이름으로 저장됐는지는 제가 볼 수 없습니다. 노트북에서 입력한 값이 PREREG.json에 들어 있을 텐데, 스크립트 파일에는 아직 `TODO_`일 수 있습니다. 해시 `eb2f284f`와 파일 내용이 같아야 하므로, 스크립트 파일에도 같은 이름을 넣고 커밋하세요.
# 2. **CHANGELOG의 v4 사유가 기록됐는지 확인하세요.** 지난번 셀을 실행하지 않았다면 지금 실행하세요(동결 후 기록이라 날짜가 동결 시각보다 늦게 찍히는 것은 괜찮습니다. 그대로 두는 것이 정직합니다).
# 3. **PREREG.md를 쓰세요.** 아래 문장을 포함하면 됩니다.
#    - 2%·5%·10%, 우연 적중률 0.50, `event_min_year=2000`, 큰 사건 규칙, `exposure_test` 사양은 진단 출력을 본 뒤 정했다.
#    - 오염 레코드 269편(규칙 251편과 검토 판정, 1편 중복) 정리는 변화점·체제 결과를 보기 전에 했고, 규칙 정규식은 데이터를 본 뒤 확정했다.
#    - 2020년 이후 구간은 이미 보았으므로 사전 명시 확인이다.
#    - 논문 2는 논문 1의 결과 변수를 쓰지 않는다.
#    - 외부 대조(0.999)는 같은 출처 간 수집 일관성 점검이지 독립 검증이 아니다.

# ## 동결 결과에서 기억할 사실
# - 주 시나리오는 NONE이므로 **S5의 중심 분석은 연속 노출 시계열**(`exposure_test`)입니다. 사건 ±1년 대조는 민감도로만 씁니다. S5에서 구현할 때 사양(Spearman, 순환이동 2000회, t=2001~2025)을 그대로 따라야 합니다.
# - 큰 사건은 6개(T-ITS 2011, TR-C 2013, TR-D 2016, TR-F 2018, T-TE 2021, T-IV 2023)이고 비중 돌파 사건이라 결과 변수와 순환합니다. 민감도(S-E)로만 쓰세요.
# - pending 저널 AMAR, TR-E는 동결 파일에 pending으로 남아 있습니다. 사건에 영향이 없으므로 그대로 둡니다.
# - 동결 이후 설정이나 임계값을 바꾸면 CHANGELOG와 multiverse에 모두 기록해야 합니다.

# ## 다음 단계: S1
# 1. **`doi_fixes.csv`의 15편**을 OpenAlex 조회에서 `doi_wrong`→`doi_fixed`로 치환하세요. 틀린 DOI로 조회하면 다른 논문의 저자·국가가 붙습니다.
# 2. **F1 연도 대 Crossref 연도 불일치율**(400편 표본)을 S1.5 커버리지 감사에서 반드시 확인하세요. 변화점 연도의 신뢰도에 직결됩니다.
# 3. 번역 레코드 제외(126+12편)의 **영문 원본 중복 여부**는 S2.8에서 확인합니다.
# 4. 창이 2010년 이후로 줄면 RQ4·RQ5를 부록으로 내립니다(기존 규칙).

# S1을 시작하실 때 OpenAlex 캐시(`works_by_doi.jsonl`)에 `authorships.institutions.country_code`가 이미 들어 있는지부터 알려 주세요. 그 필드가 있으면 S1.1의 API 재수집을 줄일 수 있습니다.

In [34]:
def open_items():
    ev = PREREG["event"]; o = []
    if not ev["fixed_venues"]: o.append("event.fixed_venues")
    vn = PREREG["risk_name"]["variant_names"]
    bad = (not vn or len(vn) != PREREG["risk_name"]["n_variants"] or len(set(vn)) != len(vn)
           or any(str(x).startswith("TODO_") or len(str(x).strip()) < 3 or set(str(x).strip()) <= set(".-_ ") for x in vn))
    if bad: o.append("risk_name.variant_names")
    return o

In [35]:
PREREG["risk_name"]["variant_names"] = ["<이름1>", "<이름2>", "<이름3>", "<이름4>", "<이름5>", "<이름6>", "<이름7>"]   # 실제 정의
FORCE_PREREG = True; STAGE = "draft"
CHECKS.clear(); STATE.clear()
for n, f in (("sec0", sec0), ("sec1", sec1), ("sec2", sec2), ("sec3", sec3), ("sec4", sec4), ("sec5", sec5), ("sec6", sec6)): safe(n, f)
FORCE_PREREG = False


[0] 진단: 오귀속 검토 -> E1 충돌 -> 시나리오별 우연 적중률 -> 주 시나리오 결정
제외 pid 269편 반영(F1 111,685 -> F1X 111,416)
연속 노출 시계열 저장 (59개 연도)

(a) 창간 연도 vs 코퍼스 첫 논문 연도(제외 반영 후) 불일치 저널
 - AMAR: 창간 2014 / 코퍼스 2013 / 판정 없음(pending) -> 코퍼스가 창간보다 앞섬(검토 파일 확인)
 - RailEng: 창간 1993 / 코퍼스 2020 / 판정 2020 -> 코퍼스가 창간보다 늦음(범위 편입 시점)
 - TR-E: 창간 1972 / 코퍼스 1997 / 판정 없음(pending) -> 코퍼스가 창간보다 늦음(범위 편입 시점)
 - TRR: 창간 1963 / 코퍼스 1967 / 판정 1967 -> 코퍼스가 창간보다 늦음(범위 편입 시점)

    검토 파일 판정 현황(저널별):
decision     EXCLUDE  KEEP
venue_short               
AAP               12     0
AMAR               0    12
CACIE              0     4
ETRR               0     2
JPT                0     2
JTTE               2     0
T-ITS             18     8
TRIP               0     2
TRR                6     0
TransSci           0     8
    pending ['AMAR', 'TR-E'] -> events_override_TEMPLATE.csv 저장(entry_year_ruling=코퍼스 진입 사건으로 쓸 연도, source_url 필수)

(b) 2000~2025 매년 논문이 있는 저널(F1X 기준) 14개: ['AAP', 'CACIE', 'JTG', 'T-ITS', 'TR-A', 'TR-B', 'TR-C', 'TR-D', 

In [36]:
#